# 1. Load Data for Exploratory Analysis

This notebook explores customer, account, transaction, financial product, and geographic patterns in the financial services dataset.

The analysis focuses on identifying meaningful business patterns while distinguishing observed facts from interpretation.

In [1]:
import pandas as pd

In [2]:
client = pd.read_csv("../data/raw/client.asc", sep=";")
account = pd.read_csv("../data/raw/account.asc", sep=";")
disp = pd.read_csv("../data/raw/disp.asc", sep=";")
district = pd.read_csv("../data/raw/district.asc", sep=";")
trans = pd.read_csv("../data/raw/trans.asc", sep=";")
loan = pd.read_csv("../data/raw/loan.asc", sep=";")
card = pd.read_csv("../data/raw/card.asc", sep=";")
order = pd.read_csv("../data/raw/order.asc", sep=";")

C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\1487775742.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  trans = pd.read_csv("../data/raw/trans.asc", sep=";")


# 2. Prepare Analytical Columns

The raw tables are prepared for exploratory analysis by standardizing key date and categorical fields used in the analysis.

The original source columns are retained, while analytical columns are created where translation or transformation is required.

In [3]:
account["account_open_date"] = pd.to_datetime(
    account["date"].astype(str),
    format="%y%m%d"
)

trans["transaction_date"] = pd.to_datetime(
    trans["date"].astype(str),
    format="%y%m%d"
)

loan["loan_date"] = pd.to_datetime(
    loan["date"].astype(str),
    format="%y%m%d"
)

card["card_issue_date"] = pd.to_datetime(
    card["issued"],
    format="%y%m%d %H:%M:%S"
)

In [4]:
transaction_type_mapping = {
    "PRIJEM": "Credit",
    "VYDAJ": "Withdrawal",
    "VYBER": "Cash Withdrawal"
}

trans["transaction_type_english"] = (
    trans["type"].map(transaction_type_mapping)
)

In [5]:
print("Transaction date range:")
print(trans["transaction_date"].min(), "to", trans["transaction_date"].max())

print("\nTransaction types:")
print(trans["transaction_type_english"].value_counts())

Transaction date range:
1993-01-01 00:00:00 to 1998-12-31 00:00:00

Transaction types:
transaction_type_english
Withdrawal         634571
Credit             405083
Cash Withdrawal     16666
Name: count, dtype: int64


# 3. Customer & Account Overview

This section provides a high-level view of the customer and account base represented in the dataset.

The analysis examines the number of customers and accounts and the distribution of account transaction-frequency settings.

In [6]:
customer_count = client["client_id"].nunique()
account_count = account["account_id"].nunique()

print("Number of customers:", customer_count)
print("Number of accounts:", account_count)

Number of customers: 5369
Number of accounts: 4500


In [7]:
account_frequency_mapping = {
    "POPLATEK MESICNE": "Monthly",
    "POPLATEK TYDNE": "Weekly",
    "POPLATEK PO OBRATU": "After Transaction"
}

account["frequency_english"] = account["frequency"].map(
    account_frequency_mapping
)

account_frequency = (
    account["frequency_english"]
    .value_counts()
    .reset_index()
)

account_frequency.columns = [
    "frequency",
    "account_count"
]

account_frequency

,frequency,account_count
0,Monthly,4167
1,Weekly,240
2,After Transaction,93


### 3.1 Account Frequency Distribution

The distribution of account frequency settings is expressed as a percentage of all accounts to provide a comparable view of the account base.

In [8]:
account_frequency["percentage_of_accounts"] = (
    account_frequency["account_count"]
    / account_count
    * 100
).round(2)

account_frequency

,frequency,account_count,percentage_of_accounts
0,Monthly,4167,92.60
1,Weekly,240,5.33
2,After Transaction,93,2.07


### 3.2 Customer-to-Account Relationship

The number of customers associated with each account is examined to understand the structure of individual and shared accounts.

In [9]:
customers_per_account = (
    disp
    .groupby("account_id")["client_id"]
    .nunique()
    .value_counts()
    .sort_index()
    .reset_index()
)

customers_per_account.columns = [
    "number_of_customers",
    "account_count"
]

customers_per_account

,number_of_customers,account_count
0,1,3631
1,2,869


### 3.3 Individual and Shared Accounts

The proportion of accounts associated with one versus two customers is calculated to quantify the prevalence of shared account relationships.

In [10]:
customers_per_account["percentage_of_accounts"] = (
    customers_per_account["account_count"]
    / account_count
    * 100
).round(2)

customers_per_account

,number_of_customers,account_count,percentage_of_accounts
0,1,3631,80.69
1,2,869,19.31


### 3.4 Customer Relationship Type

The distribution of customer relationship types is examined to distinguish primary account owners from additional account-associated customers.

In [11]:
customer_relationship_type = (
    disp["type"]
    .value_counts()
    .reset_index()
)

customer_relationship_type.columns = [
    "relationship_type",
    "customer_count"
]

customer_relationship_type

,relationship_type,customer_count
0,OWNER,4500
1,DISPONENT,869


### 3.5 Account Opening Trend

The number of accounts opened is examined by year to understand the timing and development of the account base represented in the dataset.

In [12]:
account_openings_by_year = (
    account["account_open_date"]
    .dt.year
    .value_counts()
    .sort_index()
    .reset_index()
)

account_openings_by_year.columns = [
    "year",
    "account_count"
]

account_openings_by_year

,year,account_count
0,1993,1139
1,1994,439
2,1995,661
3,1996,1363
4,1997,898


### 3.6 Customer Distribution by District

The customer base is examined across districts to understand the geographic distribution of customers represented in the dataset.

In [13]:
customer_by_district = (
    client
    .groupby("district_id")
    .size()
    .reset_index(name="customer_count")
    .sort_values("customer_count", ascending=False)
)

customer_by_district.head(10)

,district_id,customer_count
0,1,663
73,74,180
69,70,169
53,54,155
63,64,109
71,72,104
67,68,86
45,46,76
51,52,73
4,5,71


### 3.7 Customer Distribution with District Attributes

Customer counts are combined with district-level demographic and geographic attributes to provide additional context for the observed customer distribution.

In [14]:
district.columns = [
    "district_id",
    "district_name",
    "region",
    "population",
    "municipalities_under_2000",
    "municipalities_2000_9999",
    "municipalities_10000_99999",
    "municipalities_over_100000",
    "number_of_cities",
    "urban_population_pct",
    "average_salary",
    "unemployment_rate_1995",
    "unemployment_rate_1996",
    "entrepreneurs_per_1000",
    "crimes_1995",
    "crimes_1996"
]

In [15]:
customer_by_district_detail = (
    customer_by_district
    .merge(
        district[
            [
                "district_id",
                "district_name",
                "region",
                "population",
                "average_salary"
            ]
        ],
        on="district_id",
        how="left"
    )
    .sort_values("customer_count", ascending=False)
)

customer_by_district_detail.head(10)

,district_id,customer_count,district_name,region,population,average_salary
0,1,663,Hl.m. Praha,Prague,1204953,12541
1,74,180,Ostrava - mesto,north Moravia,323870,10673
2,70,169,Karvina,north Moravia,285387,10177
3,54,155,Brno - mesto,south Moravia,387570,9897
4,64,109,Zlin,south Moravia,197099,9624
5,72,104,Olomouc,north Moravia,226122,8994
6,68,86,Frydek - Mistek,north Moravia,228848,9893
7,46,76,Nachod,east Bohemia,112709,8369
8,52,73,Usti nad Orlici,east Bohemia,139012,8363
9,5,71,Kolin,central Bohemia,95616,9307


### 3.8 Customer Coverage per 1,000 Residents

Customer counts are normalized by district population to compare customer representation across districts of different population sizes.

This metric represents the number of customers in the dataset per 1,000 district residents.

In [16]:
customer_by_district_detail["customers_per_1000_residents"] = (
    customer_by_district_detail["customer_count"]
    / customer_by_district_detail["population"]
    * 1000
).round(2)

customer_by_district_detail.sort_values(
    "customers_per_1000_residents",
    ascending=False
).head(10)

,district_id,customer_count,district_name,region,population,average_salary,customers_per_1000_residents
25,69,61,Jesenik,north Moravia,42821,8173,1.42
13,19,66,Prachatice,south Bohemia,51428,8402,1.28
46,29,54,Rokycany,west Bohemia,45714,8843,1.18
32,31,59,Tachov,west Bohemia,51313,8930,1.15
19,15,63,Cesky Krumlov,south Bohemia,58796,9045,1.07
52,13,53,Rakovnik,central Bohemia,53921,8598,0.98
56,22,51,Domazlice,west Bohemia,58400,8620,0.87
28,18,60,Pisek,south Bohemia,70699,8968,0.85
18,3,63,Beroun,central Bohemia,75232,8980,0.84
41,27,55,Plzen - jih,west Bohemia,67298,8561,0.82


### 3.9 Customer Representation and Average Salary

The relationship between customer representation per 1,000 residents and district average salary is examined to identify whether the two variables show an observable association.

In [17]:
customer_salary_relationship = customer_by_district_detail[
    [
        "district_name",
        "region",
        "average_salary",
        "customers_per_1000_residents"
    ]
].copy()

customer_salary_relationship.head(10)

,district_name,region,average_salary,customers_per_1000_residents
0,Hl.m. Praha,Prague,12541,0.55
1,Ostrava - mesto,north Moravia,10673,0.56
2,Karvina,north Moravia,10177,0.59
3,Brno - mesto,south Moravia,9897,0.40
4,Zlin,south Moravia,9624,0.55
5,Olomouc,north Moravia,8994,0.46
6,Frydek - Mistek,north Moravia,9893,0.38
7,Nachod,east Bohemia,8369,0.67
8,Usti nad Orlici,east Bohemia,8363,0.53
9,Kolin,central Bohemia,9307,0.74


### 3.10 Correlation Between Average Salary and Customer Representation

The Pearson correlation coefficient is calculated across all districts to quantify the linear association between average salary and customer representation per 1,000 residents.

Correlation indicates association and does not imply causation.

In [18]:
salary_customer_correlation = (
    customer_by_district_detail[
        [
            "average_salary",
            "customers_per_1000_residents"
        ]
    ]
    .corr()
    .loc[
        "average_salary",
        "customers_per_1000_residents"
    ]
)

salary_customer_correlation

np.float64(-0.2366806433026175)

### 3.11 Understanding the Customer Birth-Date Field

The `birth_number` field is inspected to understand its encoding before creating customer demographic features.

The field contains a six-digit representation of year and month/day, with the month component using an additional encoding for female customers.

In [19]:
client["birth_number"].head(10)

0    706213
1    450204
2    406009
3    561201
4    605703
5    190922
6    290125
7    385221
8    351016
9    430501
Name: birth_number, dtype: int64

In [20]:
client["birth_year_2digit"] = client["birth_number"] // 10000

client["birth_month_encoded"] = (
    client["birth_number"] // 100
) % 100

client["birth_day"] = client["birth_number"] % 100

client[
    [
        "birth_number",
        "birth_year_2digit",
        "birth_month_encoded",
        "birth_day"
    ]
].head(10)

,birth_number,birth_year_2digit,birth_month_encoded,birth_day
0,706213,70,62,13
1,450204,45,2,4
2,406009,40,60,9
3,561201,56,12,1
4,605703,60,57,3
5,190922,19,9,22
6,290125,29,1,25
7,385221,38,52,21
8,351016,35,10,16
9,430501,43,5,1


In [21]:
client["birth_month"] = client["birth_month_encoded"].apply(
    lambda x: x - 50 if x > 50 else x
)

client["gender"] = client["birth_month_encoded"].apply(
    lambda x: "Female" if x > 50 else "Male"
)

client[
    [
        "birth_number",
        "birth_month_encoded",
        "birth_month",
        "gender"
    ]
].head(10)

,birth_number,birth_month_encoded,birth_month,gender
0,706213,62,12,Female
1,450204,2,2,Male
2,406009,60,10,Female
3,561201,12,12,Male
4,605703,57,7,Female
5,190922,9,9,Male
6,290125,1,1,Male
7,385221,52,2,Female
8,351016,10,10,Male
9,430501,5,5,Male


In [22]:
client["birth_month"].value_counts().sort_index()

birth_month
1     454
2     428
3     427
4     448
5     467
6     412
7     465
8     470
9     469
10    466
11    426
12    437
Name: count, dtype: int64

In [23]:
client["gender"].value_counts()

gender
Male      2724
Female    2645
Name: count, dtype: int64

In [24]:
client["birth_year"] = 1900 + client["birth_year_2digit"]

client["birth_date"] = pd.to_datetime(
    dict(
        year=client["birth_year"],
        month=client["birth_month"],
        day=client["birth_day"]
    )
)

client[
    [
        "birth_number",
        "birth_year",
        "birth_month",
        "birth_day",
        "birth_date",
        "gender"
    ]
].head(10)

,birth_number,birth_year,birth_month,birth_day,birth_date,gender
0,706213,1970,12,13,1970-12-13,Female
1,450204,1945,2,4,1945-02-04,Male
2,406009,1940,10,9,1940-10-09,Female
3,561201,1956,12,1,1956-12-01,Male
4,605703,1960,7,3,1960-07-03,Female
5,190922,1919,9,22,1919-09-22,Male
6,290125,1929,1,25,1929-01-25,Male
7,385221,1938,2,21,1938-02-21,Female
8,351016,1935,10,16,1935-10-16,Male
9,430501,1943,5,1,1943-05-01,Male


In [25]:
client["birth_date"].agg(["min", "max"])

min   1911-08-20
max   1987-09-27
Name: birth_date, dtype: datetime64[ns]

In [26]:
reference_date = pd.Timestamp("1998-12-31")

client["age_at_reference"] = (
    reference_date.year
    - client["birth_date"].dt.year
    - (
        (client["birth_date"].dt.month > reference_date.month)
        |
        (
            (client["birth_date"].dt.month == reference_date.month)
            & (client["birth_date"].dt.day > reference_date.day)
        )
    ).astype(int)
)

client[
    [
        "birth_date",
        "gender",
        "age_at_reference"
    ]
].head(10)

,birth_date,gender,age_at_reference
0,1970-12-13,Female,28
1,1945-02-04,Male,53
2,1940-10-09,Female,58
3,1956-12-01,Male,42
4,1960-07-03,Female,38
5,1919-09-22,Male,79
6,1929-01-25,Male,69
7,1938-02-21,Female,60
8,1935-10-16,Male,63
9,1943-05-01,Male,55


In [27]:
client["age_at_reference"].agg(["min", "max", "mean", "median"])

min       11.000000
max       87.000000
mean      44.801639
median    44.000000
Name: age_at_reference, dtype: float64

In [28]:
client.loc[
    client["age_at_reference"] < 18,
    ["client_id", "birth_date", "age_at_reference", "gender"]
].sort_values("age_at_reference")

,client_id,birth_date,age_at_reference,gender
3244,3428,1987-09-27,11,Female
4122,4354,1986-08-13,12,Male
701,734,1985-01-14,13,Female
3234,3417,1985-03-18,13,Female
5081,10201,1985-10-19,13,Male
...,...,...,...,...
4058,4289,1981-11-23,17,Male
4080,4311,1981-04-12,17,Male
4606,4976,1981-01-18,17,Male
4623,5072,1981-03-24,17,Female


In [29]:
under_18 = (
    client.loc[client["age_at_reference"] < 18, "age_at_reference"]
    .value_counts()
    .sort_index()
    .reset_index()
)

under_18.columns = [
    "age",
    "customer_count"
]

under_18

,age,customer_count
0,11,1
1,12,1
2,13,3
3,14,1
4,15,3
5,16,33
6,17,30


In [30]:
age_bins = [
    0,
    18,
    25,
    35,
    45,
    55,
    65,
    float("inf")
]

age_labels = [
    "Under 18",
    "18–24",
    "25–34",
    "35–44",
    "45–54",
    "55–64",
    "65+"
]

client["age_group"] = pd.cut(
    client["age_at_reference"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

client["age_group"].value_counts().reindex(age_labels)

age_group
Under 18     72
18–24       730
25–34       962
35–44       952
45–54       982
55–64       853
65+         818
Name: count, dtype: int64

In [31]:
age_group_distribution = (
    client["age_group"]
    .value_counts()
    .reindex(age_labels)
    .reset_index()
)

age_group_distribution.columns = [
    "age_group",
    "customer_count"
]

age_group_distribution["percentage_of_customers"] = (
    age_group_distribution["customer_count"]
    / customer_count
    * 100
).round(2)

age_group_distribution

,age_group,customer_count,percentage_of_customers
0,Under 18,72,1.34
1,18–24,730,13.60
2,25–34,962,17.92
3,35–44,952,17.73
4,45–54,982,18.29
5,55–64,853,15.89
6,65+,818,15.24


In [32]:
age_25_54_percentage = (
    age_group_distribution.loc[
        age_group_distribution["age_group"].isin(
            ["25–34", "35–44", "45–54"]
        ),
        "percentage_of_customers"
    ].sum()
)

age_25_54_percentage

np.float64(53.940000000000005)

### 3.11 Transaction Activity per Account

Transaction activity is examined at the account level to understand how frequently accounts are used.

Account-level analysis counts each transaction once and avoids duplicating transactions across multiple customers associated with a shared account.

In [33]:
transactions_per_account = (
    trans
    .groupby("account_id")
    .size()
    .reset_index(name="transaction_count")
)

transactions_per_account.head()

,account_id,transaction_count
0,1,239
1,2,478
2,3,117
3,4,186
4,5,84


In [34]:
transactions_per_account["transaction_count"].agg(
    ["min", "max", "mean", "median"]
)

min         9.000000
max       675.000000
mean      234.737778
median    208.000000
Name: transaction_count, dtype: float64

In [35]:
transactions_per_account["transaction_count"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25    133.00
0.50    208.00
0.75    330.00
0.90    411.00
0.95    459.00
0.99    565.01
Name: transaction_count, dtype: float64

In [36]:
percentage_above_330 = (
    (transactions_per_account["transaction_count"] > 330).sum()
    / account_count
    * 100
)

percentage_above_330

np.float64(24.866666666666667)

### 3.12 Transaction Value per Account

Transaction value is examined at the account level to understand the monetary volume associated with account activity.

Total transaction value is calculated by summing transaction amounts for each account. This metric measures transaction volume in monetary terms and should be interpreted separately from transaction count.

In [37]:
transaction_value_per_account = (
    trans
    .groupby("account_id")["amount"]
    .sum()
    .reset_index(name="total_transaction_value")
)

transaction_value_per_account.head()

,account_id,total_transaction_value
0,1,375174.5
1,2,3151479.3
2,3,295021.8
3,4,350971.4
4,5,166881.4


In [38]:
transaction_value_per_account["total_transaction_value"].agg(
    ["min", "max", "mean", "median"]
)

min       2.940000e+04
max       7.619102e+06
mean      1.390621e+06
median    8.918557e+05
Name: total_transaction_value, dtype: float64

### 3.13 Transaction Activity and Transaction Value

Transaction count and total transaction value are compared at the account level to examine whether more active accounts also process higher transaction values.

The relationship is exploratory and does not imply causation.

In [39]:
account_activity_value = transactions_per_account.merge(
    transaction_value_per_account,
    on="account_id",
    how="inner"
)

account_activity_value.head()

,account_id,transaction_count,total_transaction_value
0,1,239,375174.5
1,2,478,3151479.3
2,3,117,295021.8
3,4,186,350971.4
4,5,84,166881.4


In [40]:
activity_value_correlation = (
    account_activity_value[
        [
            "transaction_count",
            "total_transaction_value"
        ]
    ]
    .corr()
    .loc[
        "transaction_count",
        "total_transaction_value"
    ]
)

activity_value_correlation

np.float64(0.6107392776852588)

In [41]:
account_activity_value["average_transaction_value"] = (
    account_activity_value["total_transaction_value"]
    / account_activity_value["transaction_count"]
)

account_activity_value.head()

,account_id,transaction_count,total_transaction_value,average_transaction_value
0,1,239,375174.5,1569.767782
1,2,478,3151479.3,6593.052929
2,3,117,295021.8,2521.553846
3,4,186,350971.4,1886.943011
4,5,84,166881.4,1986.683333


In [42]:
account_activity_value["average_transaction_value"].agg(
    ["min", "max", "mean", "median"]
)

min         843.435519
max       21062.613287
mean       5757.690057
median     4729.716559
Name: average_transaction_value, dtype: float64

In [43]:
account_activity_value["average_transaction_value"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25     2303.205007
0.50     4729.716559
0.75     8352.158213
0.90    11858.017361
0.95    13587.956963
0.99    16790.469585
Name: average_transaction_value, dtype: float64

### 3.14 Transaction Activity by Account Frequency

Transaction activity is compared across account frequency settings to examine whether accounts with different recorded frequency configurations show different levels of transaction activity.

The analysis describes observed differences and does not imply that account frequency causes higher or lower transaction activity.

In [44]:
account_activity_frequency = (
    account_activity_value
    .merge(
        account[
            [
                "account_id",
                "frequency_english"
            ]
        ],
        on="account_id",
        how="left"
    )
)

account_activity_frequency.head()

,account_id,transaction_count,total_transaction_value,average_transaction_value,frequency_english
0,1,239,375174.5,1569.767782,Monthly
1,2,478,3151479.3,6593.052929,Monthly
2,3,117,295021.8,2521.553846,Monthly
3,4,186,350971.4,1886.943011,Monthly
4,5,84,166881.4,1986.683333,Monthly


In [45]:
activity_by_frequency = (
    account_activity_frequency
    .groupby("frequency_english", observed=True)
    .agg(
        account_count=("account_id", "count"),
        average_transaction_count=("transaction_count", "mean"),
        median_transaction_count=("transaction_count", "median"),
        average_transaction_value=("total_transaction_value", "mean"),
        median_transaction_value=("total_transaction_value", "median"),
        average_transaction_amount=("average_transaction_value", "mean")
    )
    .reset_index()
)

activity_by_frequency

,frequency_english,account_count,average_transaction_count,median_transaction_count,average_transaction_value,median_transaction_value,average_transaction_amount
0,After Transaction,93,263.440860,237.0,2.817198e+06,2542008.4,10948.059659
1,Monthly,4167,232.602112,206.0,1.279245e+06,820824.6,5367.438709
2,Weekly,240,260.695833,238.5,2.771584e+06,2460290.6,10522.160858


In [46]:
average_amount_by_frequency = (
    account_activity_frequency
    .groupby("frequency_english", observed=True)
    .agg(
        account_count=("account_id", "count"),
        mean_average_transaction_amount=("average_transaction_value", "mean"),
        median_average_transaction_amount=("average_transaction_value", "median")
    )
    .reset_index()
)

average_amount_by_frequency

,frequency_english,account_count,mean_average_transaction_amount,median_average_transaction_amount
0,After Transaction,93,10948.059659,11228.382273
1,Monthly,4167,5367.438709,4223.144068
2,Weekly,240,10522.160858,10420.450543


### 3.15 Transaction Activity by Customer Age Group

Transaction activity is examined across customer age groups to identify differences in account usage.

Customer age is measured at the reference date of December 31, 1998. Transaction activity is attributed through the customer-to-account relationship.

For shared accounts, transactions are attributed to the account owner only to avoid duplicating account-level transactions across associated customers.

In [47]:
owner_accounts = (
    disp[disp["type"] == "OWNER"][
        ["client_id", "account_id"]
    ]
)

owner_accounts.head()

,client_id,account_id
0,1,1
1,2,2
3,4,3
5,6,4
6,7,5


In [48]:
customer_transaction_activity = (
    owner_accounts
    .merge(
        transactions_per_account,
        on="account_id",
        how="left"
    )
)

customer_transaction_activity.head()

,client_id,account_id,transaction_count
0,1,1,239
1,2,2,478
2,4,3,117
3,6,4,186
4,7,5,84


In [49]:
customer_transaction_activity = (
    customer_transaction_activity
    .merge(
        client[
            [
                "client_id",
                "age_at_reference",
                "age_group"
            ]
        ],
        on="client_id",
        how="left"
    )
)

customer_transaction_activity.head()

,client_id,account_id,transaction_count,age_at_reference,age_group
0,1,1,239,28,25–34
1,2,2,478,53,45–54
2,4,3,117,42,35–44
3,6,4,186,79,65+
4,7,5,84,69,65+


In [50]:
transaction_activity_by_age = (
    customer_transaction_activity
    .groupby("age_group", observed=True)
    .agg(
        customer_count=("client_id", "count"),
        average_transaction_count=("transaction_count", "mean"),
        median_transaction_count=("transaction_count", "median")
    )
    .reset_index()
)

transaction_activity_by_age

,age_group,customer_count,average_transaction_count,median_transaction_count
0,Under 18,57,223.754386,209.0
1,18–24,658,230.214286,205.5
2,25–34,819,242.627595,217.0
3,35–44,798,243.392231,213.0
4,45–54,832,247.090144,221.0
5,55–64,697,233.562410,205.0
6,65+,639,204.654147,182.0


### 3.16 Transaction Value by Customer Age Group

Total transaction value is examined across customer age groups to compare monetary transaction volume associated with customer-owned accounts.

Transaction value is calculated as the sum of transaction amounts recorded for each owned account. The metric represents total transaction amount and does not distinguish between incoming and outgoing transactions.

In [51]:
customer_transaction_value = (
    owner_accounts
    .merge(
        transaction_value_per_account,
        on="account_id",
        how="left"
    )
    .merge(
        client[
            [
                "client_id",
                "age_group"
            ]
        ],
        on="client_id",
        how="left"
    )
)

customer_transaction_value.head()

,client_id,account_id,total_transaction_value,age_group
0,1,1,375174.5,25–34
1,2,2,3151479.3,45–54
2,4,3,295021.8,35–44
3,6,4,350971.4,65+
4,7,5,166881.4,65+


In [52]:
transaction_value_by_age = (
    customer_transaction_value
    .groupby("age_group", observed=True)
    .agg(
        customer_count=("client_id", "count"),
        average_transaction_value=("total_transaction_value", "mean"),
        median_transaction_value=("total_transaction_value", "median")
    )
    .reset_index()
)

transaction_value_by_age

,age_group,customer_count,average_transaction_value,median_transaction_value
0,Under 18,57,5.843799e+05,577666.30
1,18–24,658,1.380864e+06,886232.20
2,25–34,819,1.621188e+06,1224125.70
3,35–44,798,1.631702e+06,1249581.25
4,45–54,832,1.707105e+06,1271996.70
5,55–64,697,1.419189e+06,958831.80
6,65+,639,4.327658e+05,381145.30


In [53]:
customer_transaction_analysis = (
    customer_transaction_activity[
        [
            "client_id",
            "account_id",
            "age_group",
            "transaction_count"
        ]
    ]
    .merge(
        transaction_value_per_account,
        on="account_id",
        how="left"
    )
)

customer_transaction_analysis["average_transaction_amount"] = (
    customer_transaction_analysis["total_transaction_value"]
    / customer_transaction_analysis["transaction_count"]
)

customer_transaction_analysis.head()

,client_id,account_id,age_group,transaction_count,total_transaction_value,average_transaction_amount
0,1,1,25–34,239,375174.5,1569.767782
1,2,2,45–54,478,3151479.3,6593.052929
2,4,3,35–44,117,295021.8,2521.553846
3,6,4,65+,186,350971.4,1886.943011
4,7,5,65+,84,166881.4,1986.683333


In [54]:
transaction_analysis_by_age = (
    customer_transaction_analysis
    .groupby("age_group", observed=True)
    .agg(
        customer_count=("client_id", "count"),
        average_transaction_count=("transaction_count", "mean"),
        median_transaction_count=("transaction_count", "median"),
        average_total_transaction_value=("total_transaction_value", "mean"),
        median_total_transaction_value=("total_transaction_value", "median"),
        average_transaction_amount=("average_transaction_amount", "mean"),
        median_transaction_amount=("average_transaction_amount", "median")
    )
    .reset_index()
)

transaction_analysis_by_age

,age_group,customer_count,average_transaction_count,median_transaction_count,average_total_transaction_value,median_total_transaction_value,average_transaction_amount,median_transaction_amount
0,Under 18,57,223.754386,209.0,5.843799e+05,577666.30,2830.518414,2506.014216
1,18–24,658,230.214286,205.5,1.380864e+06,886232.20,5919.562363,4592.081155
2,25–34,819,242.627595,217.0,1.621188e+06,1224125.70,6550.020550,6040.884496
3,35–44,798,243.392231,213.0,1.631702e+06,1249581.25,6624.846507,6133.393490
4,45–54,832,247.090144,221.0,1.707105e+06,1271996.70,6864.116825,6594.121777
5,55–64,697,233.562410,205.0,1.419189e+06,958831.80,5915.468272,5202.130370
6,65+,639,204.654147,182.0,4.327658e+05,381145.30,2140.958911,2140.357107


### 3.17 Transaction Type Distribution

The distribution of transaction types is examined to understand the composition of recorded transaction activity.

Transaction types are based on the standardized categories Credit, Withdrawal, and Cash Withdrawal.

In [55]:
transaction_type_distribution = (
    trans["transaction_type_english"]
    .value_counts()
    .reset_index()
)

transaction_type_distribution.columns = [
    "transaction_type",
    "transaction_count"
]

transaction_type_distribution["percentage_of_transactions"] = (
    transaction_type_distribution["transaction_count"]
    / len(trans)
    * 100
).round(2)

transaction_type_distribution

,transaction_type,transaction_count,percentage_of_transactions
0,Withdrawal,634571,60.07
1,Credit,405083,38.35
2,Cash Withdrawal,16666,1.58


### 3.18 Transaction Value by Transaction Type

Transaction value is compared across transaction types to examine whether the composition of transaction activity differs when measured by monetary volume rather than transaction count.

Total transaction value is calculated as the sum of recorded transaction amounts within each transaction type.

In [56]:
transaction_value_by_type = (
    trans
    .groupby("transaction_type_english")
    .agg(
        transaction_count=("trans_id", "count"),
        total_transaction_value=("amount", "sum"),
        average_transaction_amount=("amount", "mean"),
        median_transaction_amount=("amount", "median")
    )
    .reset_index()
)

transaction_value_by_type

,transaction_type_english,transaction_count,total_transaction_value,average_transaction_amount,median_transaction_amount
0,Cash Withdrawal,16666,2.086038e+08,12516.726149,12103.0
1,Credit,405083,3.227472e+09,7967.434882,1600.0
2,Withdrawal,634571,2.821717e+09,4446.653531,2042.0


### 3.19 Transaction Activity Over Time

Transaction activity is examined by year to understand how the volume of recorded transactions changed over the period covered by the dataset.

The analysis focuses on transaction count and total transaction value to distinguish changes in activity from changes in monetary volume.

In [57]:
transaction_activity_by_year = (
    trans
    .groupby(trans["transaction_date"].dt.year)
    .agg(
        transaction_count=("trans_id", "count"),
        total_transaction_value=("amount", "sum")
    )
    .reset_index()
)

transaction_activity_by_year.columns = [
    "year",
    "transaction_count",
    "total_transaction_value"
]

transaction_activity_by_year

,year,transaction_count,total_transaction_value
0,1993,28205,2.043085e+08
1,1994,91628,5.387907e+08
2,1995,133022,7.677305e+08
3,1996,196779,1.163851e+09
4,1997,284409,1.688329e+09
5,1998,322277,1.894783e+09


In [58]:
transaction_activity_by_year["average_transaction_amount"] = (
    transaction_activity_by_year["total_transaction_value"]
    / transaction_activity_by_year["transaction_count"]
)

transaction_activity_by_year

,year,transaction_count,total_transaction_value,average_transaction_amount
0,1993,28205,2.043085e+08,7243.698943
1,1994,91628,5.387907e+08,5880.197437
2,1995,133022,7.677305e+08,5771.454828
3,1996,196779,1.163851e+09,5914.510534
4,1997,284409,1.688329e+09,5936.271395
5,1998,322277,1.894783e+09,5879.362652


### 3.20 Transaction Value by Year and Transaction Type

Transaction value is examined by year and transaction type to identify how the composition of monetary transaction volume changed over time.

The analysis compares total recorded transaction value across Credit, Withdrawal, and Cash Withdrawal transactions.

In [59]:
transaction_value_by_year_type = (
    trans
    .groupby(
        [
            trans["transaction_date"].dt.year,
            "transaction_type_english"
        ]
    )
    .agg(
        transaction_count=("trans_id", "count"),
        total_transaction_value=("amount", "sum")
    )
    .reset_index()
)

transaction_value_by_year_type.columns = [
    "year",
    "transaction_type",
    "transaction_count",
    "total_transaction_value"
]

transaction_value_by_year_type

,year,transaction_type,transaction_count,total_transaction_value
0,1993,Cash Withdrawal,371,4767384.0
1,1993,Credit,15212,120442237.4
2,1993,Withdrawal,12622,79098907.3
3,1994,Cash Withdrawal,1485,18256999.0
4,1994,Credit,35743,278535354.4
5,1994,Withdrawal,54400,241998377.4
6,1995,Cash Withdrawal,2045,25695422.0
7,1995,Credit,50633,396067245.1
8,1995,Withdrawal,80344,345967797.0
9,1996,Cash Withdrawal,2856,36079111.0


### 3.21 Monthly Transaction Activity

Transaction activity is examined by month to identify shorter-term patterns in transaction volume over the period covered by the dataset.

Monthly transaction counts and total transaction values are compared to identify periods of higher or lower recorded activity.

In [60]:
transaction_activity_by_month = (
    trans
    .groupby(trans["transaction_date"].dt.to_period("M"))
    .agg(
        transaction_count=("trans_id", "count"),
        total_transaction_value=("amount", "sum")
    )
    .reset_index()
)

transaction_activity_by_month["month"] = (
    transaction_activity_by_month["transaction_date"]
    .astype(str)
)

transaction_activity_by_month = (
    transaction_activity_by_month[
        [
            "month",
            "transaction_count",
            "total_transaction_value"
        ]
    ]
)

transaction_activity_by_month.head(12)

,month,transaction_count,total_transaction_value
0,1993-01,177,702157.6
1,1993-02,395,2726925.3
2,1993-03,676,4730318.6
3,1993-04,913,7378367.8
4,1993-05,1306,11680753.4
5,1993-06,1880,18291948.9
6,1993-07,2399,18021294.7
7,1993-08,2938,21028811.6
8,1993-09,3441,23438701.6
9,1993-10,3989,27570899.7


In [61]:
transaction_activity_by_month.tail(12)

,month,transaction_count,total_transaction_value
60,1998-01,42940,192082358.2
61,1998-02,23995,128749327.0
62,1998-03,25341,148188410.7
63,1998-04,25297,147273342.6
64,1998-05,25634,149884936.1
65,1998-06,27833,209902026.5
66,1998-07,25582,147337119.2
67,1998-08,25622,150639332.7
68,1998-09,25450,148297693.4
69,1998-10,25689,151889623.6


In [62]:
transaction_activity_by_month["average_transaction_amount"] = (
    transaction_activity_by_month["total_transaction_value"]
    / transaction_activity_by_month["transaction_count"]
)

transaction_activity_by_month.head(12)

,month,transaction_count,total_transaction_value,average_transaction_amount
0,1993-01,177,702157.6,3966.992090
1,1993-02,395,2726925.3,6903.608354
2,1993-03,676,4730318.6,6997.512722
3,1993-04,913,7378367.8,8081.454326
4,1993-05,1306,11680753.4,8943.915314
5,1993-06,1880,18291948.9,9729.760053
6,1993-07,2399,18021294.7,7512.002793
7,1993-08,2938,21028811.6,7157.526072
8,1993-09,3441,23438701.6,6811.595931
9,1993-10,3989,27570899.7,6911.732189


In [63]:
transaction_activity_by_month.sort_values(
    "average_transaction_amount",
    ascending=False
).head(10)

,month,transaction_count,total_transaction_value,average_transaction_amount
5,1993-06,1880,18291948.9,9729.760053
4,1993-05,1306,11680753.4,8943.915314
3,1993-04,913,7378367.8,8081.454326
53,1997-06,23832,180533012.8,7575.235515
65,1998-06,27833,209902026.5,7541.480491
17,1994-06,7916,59676368.5,7538.702438
6,1993-07,2399,18021294.7,7512.002793
41,1996-06,15885,118709525.0,7473.057916
29,1995-06,10930,81612628.1,7466.846121
71,1998-12,23553,171697093.7,7289.818439


In [64]:
transaction_activity_by_month.sort_values(
    "average_transaction_amount",
    ascending=True
).head(10)

,month,transaction_count,total_transaction_value,average_transaction_amount
0,1993-01,177,702157.6,3966.992090
24,1995-01,15493,66245579.7,4275.839392
36,1996-01,21405,93622315.9,4373.852647
60,1998-01,42940,192082358.2,4473.273363
48,1997-01,33060,149308207.7,4516.279725
12,1994-01,9139,42592232.5,4660.491575
25,1995-02,8682,45126076.7,5197.659145
61,1998-02,23995,128749327.0,5365.673140
37,1996-02,12507,67669651.0,5410.542176
49,1997-02,19202,105167650.0,5476.911259


### 3.22 Monthly Transaction Activity by Transaction Type

Monthly transaction activity is examined separately for Credit, Withdrawal, and Cash Withdrawal transactions.

The analysis compares transaction counts and total recorded transaction value across transaction types and months to identify differences in monthly activity patterns.

In [65]:
monthly_activity_by_type = (
    trans
    .groupby(
        [
            trans["transaction_date"].dt.to_period("M"),
            "transaction_type_english"
        ]
    )
    .agg(
        transaction_count=("trans_id", "count"),
        total_transaction_value=("amount", "sum")
    )
    .reset_index()
)

monthly_activity_by_type["month"] = (
    monthly_activity_by_type["transaction_date"]
    .astype(str)
)

monthly_activity_by_type = (
    monthly_activity_by_type[
        [
            "month",
            "transaction_type_english",
            "transaction_count",
            "total_transaction_value"
        ]
    ]
)

monthly_activity_by_type.head(15)

,month,transaction_type_english,transaction_count,total_transaction_value
0,1993-01,Credit,176,667457.6
1,1993-01,Withdrawal,1,34700.0
2,1993-02,Credit,372,2445498.7
3,1993-02,Withdrawal,23,281426.6
4,1993-03,Cash Withdrawal,2,24334.0
5,1993-03,Credit,586,3855629.4
6,1993-03,Withdrawal,88,850355.2
7,1993-04,Cash Withdrawal,3,40360.0
8,1993-04,Credit,746,5325748.8
9,1993-04,Withdrawal,164,2012259.0


In [66]:
monthly_activity_by_type.loc[
    monthly_activity_by_type
    .groupby("transaction_type_english")["transaction_count"]
    .idxmax()
]

,month,transaction_type_english,transaction_count,total_transaction_value
184,1998-03,Cash Withdrawal,500,6381965.0
179,1998-01,Credit,10172,79269558.3
180,1998-01,Withdrawal,32414,108509695.9


In [67]:
monthly_activity_by_type.loc[
    monthly_activity_by_type
    .groupby("transaction_type_english")["total_transaction_value"]
    .idxmax()
]

,month,transaction_type_english,transaction_count,total_transaction_value
196,1998-07,Cash Withdrawal,496,6406987.0
194,1998-06,Credit,10005,98321360.5
180,1998-01,Withdrawal,32414,108509695.9


In [68]:
monthly_activity_by_type["average_transaction_amount"] = (
    monthly_activity_by_type["total_transaction_value"]
    / monthly_activity_by_type["transaction_count"]
)

monthly_activity_by_type.head(15)

,month,transaction_type_english,transaction_count,total_transaction_value,average_transaction_amount
0,1993-01,Credit,176,667457.6,3792.372727
1,1993-01,Withdrawal,1,34700.0,34700.000000
2,1993-02,Credit,372,2445498.7,6573.921237
3,1993-02,Withdrawal,23,281426.6,12235.939130
4,1993-03,Cash Withdrawal,2,24334.0,12167.000000
5,1993-03,Credit,586,3855629.4,6579.572355
6,1993-03,Withdrawal,88,850355.2,9663.127273
7,1993-04,Cash Withdrawal,3,40360.0,13453.333333
8,1993-04,Credit,746,5325748.8,7139.073458
9,1993-04,Withdrawal,164,2012259.0,12269.871951


### Interpretation

Monthly transaction activity varies across both transaction count and total recorded transaction value.

The month with the highest transaction count does not always correspond to the month with the highest total transaction value for a given transaction type. This indicates that transaction frequency and transaction size provide different perspectives on financial activity.

Credit and Withdrawal transactions reached their highest monthly transaction counts in January 1998, while Credit reached its highest total recorded value in June 1998. Cash Withdrawal reached its highest transaction count in March 1998 but its highest total recorded value in July 1998.

Average transaction amounts should be interpreted together with transaction counts, particularly for months or transaction types with relatively few observations.

These findings are descriptive and do not establish the causes of monthly variation.

### 3.23 Financial Product Usage

Financial product usage is examined to understand how loans and cards are distributed across the accounts in the dataset.

The analysis focuses on observed product relationships rather than assuming that product ownership indicates customer profitability or financial need.

In [69]:
loan_account_count = loan["account_id"].nunique()
card_account_count = (
    card
    .merge(
        disp[["disp_id", "account_id"]],
        on="disp_id",
        how="left"
    )["account_id"]
    .nunique()
)

loan_account_count, card_account_count

(682, 892)

In [70]:
loan_account_percentage = (
    loan_account_count
    / account_count
    * 100
)

card_account_percentage = (
    card_account_count
    / account_count
    * 100
)

loan_account_percentage, card_account_percentage

(15.155555555555555, 19.822222222222223)

In [71]:
product_usage_summary = pd.DataFrame({
    "product": ["Loan", "Card"],
    "accounts_with_product": [
        loan_account_count,
        card_account_count
    ],
    "percentage_of_accounts": [
        loan_account_percentage,
        card_account_percentage
    ]
})

product_usage_summary["percentage_of_accounts"] = (
    product_usage_summary["percentage_of_accounts"]
    .round(2)
)

product_usage_summary

,product,accounts_with_product,percentage_of_accounts
0,Loan,682,15.16
1,Card,892,19.82


In [72]:
loan_accounts = set(loan["account_id"])

card_accounts = set(
    card
    .merge(
        disp[["disp_id", "account_id"]],
        on="disp_id",
        how="left"
    )["account_id"]
)

accounts_with_both = len(loan_accounts & card_accounts)
accounts_with_loan_only = len(loan_accounts - card_accounts)
accounts_with_card_only = len(card_accounts - loan_accounts)
accounts_with_neither = (
    account_count
    - len(loan_accounts | card_accounts)
)

accounts_with_both, accounts_with_loan_only, accounts_with_card_only, accounts_with_neither

(170, 512, 722, 3096)

In [73]:
product_overlap_summary = pd.DataFrame({
    "account_category": [
        "Both loan and card",
        "Loan only",
        "Card only",
        "Neither"
    ],
    "account_count": [
        accounts_with_both,
        accounts_with_loan_only,
        accounts_with_card_only,
        accounts_with_neither
    ]
})

product_overlap_summary["percentage_of_accounts"] = (
    product_overlap_summary["account_count"]
    / account_count
    * 100
).round(2)

product_overlap_summary

,account_category,account_count,percentage_of_accounts
0,Both loan and card,170,3.78
1,Loan only,512,11.38
2,Card only,722,16.04
3,Neither,3096,68.80


### 3.24 Loan Portfolio Status

The loan portfolio is examined by loan status to understand the distribution of completed and ongoing loan contracts.

Loan statuses are based on the definitions provided with the dataset. The analysis describes the observed portfolio composition and does not independently assess credit risk or loan performance beyond the available status information.

In [74]:
loan_status_mapping = {
    "A": "Finished - No Problems",
    "B": "Finished - Not Paid",
    "C": "Running - OK",
    "D": "Running - In Debt"
}

loan["loan_status_english"] = loan["status"].map(
    loan_status_mapping
)

loan_status_summary = (
    loan["loan_status_english"]
    .value_counts()
    .reset_index()
)

loan_status_summary.columns = [
    "loan_status",
    "loan_count"
]

loan_status_summary["percentage_of_loans"] = (
    loan_status_summary["loan_count"]
    / len(loan)
    * 100
).round(2)

loan_status_summary

,loan_status,loan_count,percentage_of_loans
0,Running - OK,403,59.09
1,Finished - No Problems,203,29.77
2,Running - In Debt,45,6.60
3,Finished - Not Paid,31,4.55


### 3.25 Loan Amount & Duration Analysis

Loan amounts and durations are examined to understand the financial size and repayment structure of the recorded loan portfolio.

The analysis uses descriptive statistics to identify the typical loan size and the range of loan durations. Monetary values are reported in the dataset's original units because the dataset documentation does not establish a currency for this analysis.

In [75]:
loan_portfolio_summary = pd.DataFrame({
    "metric": [
        "Number of loans",
        "Total loan amount",
        "Average loan amount",
        "Median loan amount",
        "Minimum loan amount",
        "Maximum loan amount",
        "Average duration",
        "Median duration"
    ],
    "value": [
        len(loan),
        loan["amount"].sum(),
        loan["amount"].mean(),
        loan["amount"].median(),
        loan["amount"].min(),
        loan["amount"].max(),
        loan["duration"].mean(),
        loan["duration"].median()
    ]
})

loan_portfolio_summary

,metric,value
0,Number of loans,6.820000e+02
1,Total loan amount,1.032617e+08
2,Average loan amount,1.514102e+05
3,Median loan amount,1.169280e+05
4,Minimum loan amount,4.980000e+03
5,Maximum loan amount,5.908200e+05
6,Average duration,3.649267e+01
7,Median duration,3.600000e+01


### 3.26 Loan Amount by Status

Loan amounts are compared across loan-status categories to examine whether the financial size of recorded loans differs by status.

The analysis uses average and median loan amounts to reduce the risk of interpreting a small number of large loans as representative of an entire status group.

In [76]:
loan_amount_by_status = (
    loan
    .groupby("loan_status_english")
    .agg(
        loan_count=("loan_id", "count"),
        total_loan_amount=("amount", "sum"),
        average_loan_amount=("amount", "mean"),
        median_loan_amount=("amount", "median")
    )
    .reset_index()
)

loan_amount_by_status.columns = [
    "loan_status",
    "loan_count",
    "total_loan_amount",
    "average_loan_amount",
    "median_loan_amount"
]

loan_amount_by_status

,loan_status,loan_count,total_loan_amount,average_loan_amount,median_loan_amount
0,Finished - No Problems,203,18603216,91641.458128,79632.0
1,Finished - Not Paid,31,4362348,140720.903226,96396.0
2,Running - In Debt,45,11217804,249284.533333,260400.0
3,Running - OK,403,69078372,171410.352357,153504.0


### 3.27 Loan Duration by Status

Loan duration is compared across loan-status categories to examine whether the length of recorded loan contracts differs between status groups.

Average and median duration are used together to provide a more balanced view of the typical contract length.

In [77]:
loan_duration_by_status = (
    loan
    .groupby("loan_status_english")
    .agg(
        loan_count=("loan_id", "count"),
        average_duration=("duration", "mean"),
        median_duration=("duration", "median"),
        minimum_duration=("duration", "min"),
        maximum_duration=("duration", "max")
    )
    .reset_index()
)

loan_duration_by_status.columns = [
    "loan_status",
    "loan_count",
    "average_duration",
    "median_duration",
    "minimum_duration",
    "maximum_duration"
]

loan_duration_by_status

,loan_status,loan_count,average_duration,median_duration,minimum_duration,maximum_duration
0,Finished - No Problems,203,22.226601,24.0,12,60
1,Finished - Not Paid,31,25.548387,24.0,12,60
2,Running - In Debt,45,46.133333,48.0,12,60
3,Running - OK,403,43.444169,48.0,12,60


### 3.28 Loan Amount vs. Duration

The relationship between loan amount and loan duration is examined to determine whether larger recorded loans tend to be associated with longer repayment periods.

Pearson correlation is used to measure the strength and direction of the linear association between the two variables. Correlation does not establish causation.

In [78]:
loan_amount_duration_correlation = loan[
    ["amount", "duration"]
].corr().loc["amount", "duration"]

loan_amount_duration_correlation

np.float64(0.6124018342117451)

### 3.29 Loan Status Risk Indicator

Loan statuses associated with repayment problems are grouped into a descriptive risk-indicator category based on the status definitions provided with the dataset.

The indicator is intended for portfolio monitoring and exploratory analysis only. It does not represent a formal credit-risk model or predict future default.

In [79]:
risk_statuses = [
    "Finished - Not Paid",
    "Running - In Debt"
]

loan["risk_indicator"] = loan["loan_status_english"].isin(
    risk_statuses
)

loan_risk_summary = pd.DataFrame({
    "category": [
        "Risk indicator",
        "No risk indicator"
    ],
    "loan_count": [
        loan["risk_indicator"].sum(),
        (~loan["risk_indicator"]).sum()
    ]
})

loan_risk_summary["percentage_of_loans"] = (
    loan_risk_summary["loan_count"]
    / len(loan)
    * 100
).round(2)

loan_risk_summary

,category,loan_count,percentage_of_loans
0,Risk indicator,76,11.14
1,No risk indicator,606,88.86


### 3.30 Transaction Activity of Loan-Associated Accounts

Transaction activity is compared between accounts associated with loans that have a risk indicator and those without a risk indicator.

The analysis uses transaction count and total recorded transaction value at the account level. Risk classification is based on the loan-status indicator created in the previous analysis.

The comparison is descriptive and does not establish that transaction behaviour causes or predicts loan repayment problems.

In [80]:
loan_account_activity = (
    loan[
        [
            "loan_id",
            "account_id",
            "loan_status_english",
            "risk_indicator"
        ]
    ]
    .merge(
        transactions_per_account,
        on="account_id",
        how="left"
    )
    .merge(
        transaction_value_per_account,
        on="account_id",
        how="left"
    )
)

loan_account_activity.head()

,loan_id,account_id,loan_status_english,risk_indicator,transaction_count,total_transaction_value
0,5314,1787,Finished - Not Paid,True,403,4108891.9
1,5316,1801,Finished - No Problems,False,633,4751818.5
2,6863,9188,Finished - No Problems,False,605,1961962.3
3,5325,1843,Finished - No Problems,False,462,2764436.9
4,7240,11013,Finished - No Problems,False,571,6376283.8


In [81]:
loan_account_activity.shape

(682, 6)

In [82]:
loan_account_activity["loan_id"].nunique()

682

In [83]:
loan_account_activity["transaction_count"].isna().sum()

np.int64(0)

In [84]:
loan_activity_by_risk = (
    loan_account_activity
    .groupby("risk_indicator")
    .agg(
        loan_count=("loan_id", "count"),
        average_transaction_count=("transaction_count", "mean"),
        median_transaction_count=("transaction_count", "median"),
        average_total_transaction_value=("total_transaction_value", "mean"),
        median_total_transaction_value=("total_transaction_value", "median")
    )
    .reset_index()
)

loan_activity_by_risk["risk_category"] = (
    loan_activity_by_risk["risk_indicator"]
    .map({
        True: "Risk Indicator",
        False: "No Risk Indicator"
    })
)

loan_activity_by_risk = loan_activity_by_risk[
    [
        "risk_category",
        "loan_count",
        "average_transaction_count",
        "median_transaction_count",
        "average_total_transaction_value",
        "median_total_transaction_value"
    ]
]

loan_activity_by_risk

,risk_category,loan_count,average_transaction_count,median_transaction_count,average_total_transaction_value,median_total_transaction_value
0,No Risk Indicator,606,278.541254,247.5,2.301545e+06,1980898.45
1,Risk Indicator,76,299.473684,286.5,2.509361e+06,2355679.50


### 3.31 Card Portfolio by Card Type

The card portfolio is examined by card type to understand the composition of cards recorded in the dataset.

The analysis describes the observed distribution of card types and does not infer customer value, profitability, or product performance from card type alone.

In [85]:
card_type_summary = (
    card["type"]
    .value_counts()
    .reset_index()
)

card_type_summary.columns = [
    "card_type",
    "card_count"
]

card_type_summary["percentage_of_cards"] = (
    card_type_summary["card_count"]
    / len(card)
    * 100
).round(2)

card_type_summary

,card_type,card_count,percentage_of_cards
0,classic,659,73.88
1,junior,145,16.26
2,gold,88,9.87


### 3.32 Transaction Activity by Card Type

Transaction activity is compared across card types by linking cards to their associated accounts through the disposition table.

The analysis examines transaction count and total recorded transaction value at the account level.

Because an account may have more than one card, the analysis should account for potential duplication when interpreting account-level transaction activity.

In [86]:
card_account = (
    card[
        [
            "card_id",
            "disp_id",
            "type"
        ]
    ]
    .merge(
        disp[
            [
                "disp_id",
                "account_id"
            ]
        ],
        on="disp_id",
        how="left"
    )
)

card_account.head()

,card_id,disp_id,type,account_id
0,1005,9285,classic,7753
1,104,588,classic,489
2,747,4915,classic,4078
3,70,439,classic,361
4,577,3687,classic,3050


In [87]:
card_account["account_id"].nunique()

892

In [88]:
card_account["card_id"].nunique()

892

In [89]:
card_account_activity = (
    card_account
    .merge(
        transactions_per_account,
        on="account_id",
        how="left"
    )
    .merge(
        transaction_value_per_account,
        on="account_id",
        how="left"
    )
)

card_account_activity.head()

,card_id,disp_id,type,account_id,transaction_count,total_transaction_value
0,1005,9285,classic,7753,500,6346623.2
1,104,588,classic,489,382,2367275.5
2,747,4915,classic,4078,321,3580288.8
3,70,439,classic,361,468,3590897.7
4,577,3687,classic,3050,609,6079276.8


In [90]:
card_account_activity.shape

(892, 6)

In [91]:
card_account_activity["card_id"].nunique()

892

In [92]:
card_account_activity["transaction_count"].isna().sum()

np.int64(0)

In [93]:
card_activity_by_type = (
    card_account_activity
    .groupby("type")
    .agg(
        card_count=("card_id", "count"),
        average_transaction_count=("transaction_count", "mean"),
        median_transaction_count=("transaction_count", "median"),
        average_total_transaction_value=("total_transaction_value", "mean"),
        median_total_transaction_value=("total_transaction_value", "median")
    )
    .reset_index()
)

card_activity_by_type.columns = [
    "card_type",
    "card_count",
    "average_transaction_count",
    "median_transaction_count",
    "average_total_transaction_value",
    "median_total_transaction_value"
]

card_activity_by_type

,card_type,card_count,average_transaction_count,median_transaction_count,average_total_transaction_value,median_total_transaction_value
0,classic,659,242.578149,216.0,2.093472e+06,1718716.90
1,gold,88,292.500000,298.0,2.664180e+06,2369136.35
2,junior,145,250.613793,227.0,1.913814e+06,1452207.90


### Interpretation

Transaction activity differs across card types in the dataset.

Gold cards are associated with the highest average transaction count (292.50) and the highest average total transaction value (approximately 2.66 million in the dataset's original units). Classic cards represent the largest card group, with 659 cards, but show lower average transaction activity than Gold cards. Junior cards show intermediate transaction activity but the lowest average total transaction value.

The median values show a similar pattern, with Gold remaining highest for both transaction count and total transaction value.

These results describe transaction activity associated with card types and should not be interpreted as evidence of profitability or superior product performance. Additional information such as fees, revenue, costs, and customer-level product usage would be required for a profitability assessment. The smaller number of Gold cards should also be considered when interpreting the results.

### 3.33 Card Issuance Over Time

Card issuance is examined by year to understand how the card portfolio developed over the period covered by the dataset.

The analysis focuses on the number of cards issued each year and compares issuance across card types. The results describe observed issuance activity and do not establish the reasons for changes over time.

In [94]:
card_issuance_by_year = (
    card
    .groupby(card["card_issue_date"].dt.year)
    .agg(
        card_count=("card_id", "count")
    )
    .reset_index()
)

card_issuance_by_year.columns = [
    "year",
    "card_count"
]

card_issuance_by_year

,year,card_count
0,1993,1
1,1994,21
2,1995,63
3,1996,116
4,1997,242
5,1998,449


In [95]:
card_issuance_by_year_type = (
    card
    .groupby(
        [
            card["card_issue_date"].dt.year,
            "type"
        ]
    )
    .agg(
        card_count=("card_id", "count")
    )
    .reset_index()
)

card_issuance_by_year_type.columns = [
    "year",
    "card_type",
    "card_count"
]

card_issuance_by_year_type

,year,card_type,card_count
0,1993,classic,1
1,1994,classic,17
2,1994,junior,4
3,1995,classic,42
4,1995,gold,4
5,1995,junior,17
6,1996,classic,83
7,1996,gold,5
8,1996,junior,28
9,1997,classic,190


### Interpretation

Card issuance increased consistently throughout the observed period, reaching 449 newly issued cards in 1998 compared with 1 card in 1993.

Classic cards accounted for the largest number of issuances in every year and represented the main component of the card portfolio's expansion. Junior and Gold card issuances also increased over time, particularly in 1998.

The results describe the observed development of card issuance within the dataset. They do not establish the business reasons for the increase or indicate changes in overall market demand.

In [96]:
card_issuance_composition = (
    card_issuance_by_year_type
    .merge(
        card_issuance_by_year,
        on="year",
        how="left",
        suffixes=("", "_total")
    )
)

card_issuance_composition["percentage_of_yearly_issuance"] = (
    card_issuance_composition["card_count"]
    / card_issuance_composition["card_count_total"]
    * 100
).round(2)

card_issuance_composition

,year,card_type,card_count,card_count_total,percentage_of_yearly_issuance
0,1993,classic,1,1,100.00
1,1994,classic,17,21,80.95
2,1994,junior,4,21,19.05
3,1995,classic,42,63,66.67
4,1995,gold,4,63,6.35
5,1995,junior,17,63,26.98
6,1996,classic,83,116,71.55
7,1996,gold,5,116,4.31
8,1996,junior,28,116,24.14
9,1997,classic,190,242,78.51


### Interpretation

Classic cards remained the dominant card type in yearly issuance throughout the observed period.

The share of Classic cards varied across years, ranging from 66.67% in 1995 to 80.95% in 1994, excluding 1993 because only one card was issued that year. In 1998, Classic cards represented 72.61% of yearly issuance, followed by Junior cards at 15.59% and Gold cards at 11.80%.

Gold cards represented a larger share of yearly issuance in 1998 than in 1996, increasing from 4.31% to 11.80%. Over the same period, the share of Junior cards decreased from 24.14% to 15.59%.

These results describe changes in the composition of recorded card issuance. They do not provide evidence about customer preferences, profitability, or the reasons for changes in product mix.

In [97]:
customer_card = (
    disp[
        [
            "client_id",
            "disp_id",
            "type"
        ]
    ]
    .merge(
        card[
            [
                "card_id",
                "disp_id",
                "type"
            ]
        ],
        on="disp_id",
        how="inner",
        suffixes=("_disp", "_card")
    )
)

customer_card.head()

,client_id,disp_id,type_disp,card_id,type_card
0,9,9,OWNER,1,gold
1,19,19,OWNER,2,classic
2,41,41,OWNER,3,gold
3,42,42,OWNER,4,classic
4,51,51,OWNER,5,junior


In [98]:
customer_card.shape

(892, 5)

In [99]:
customer_card["card_id"].nunique()

892

In [100]:
customer_card["client_id"].nunique()

892

In [101]:
customer_card["card_id"].duplicated().sum()

np.int64(0)

In [102]:
customer_card_age = (
    customer_card
    .merge(
        client[
            [
                "client_id",
                "age_group"
            ]
        ],
        on="client_id",
        how="left"
    )
)

customer_card_age.head()

,client_id,disp_id,type_disp,card_id,type_card,age_group
0,9,9,OWNER,1,gold,55–64
1,19,19,OWNER,2,classic,55–64
2,41,41,OWNER,3,gold,25–34
3,42,42,OWNER,4,classic,55–64
4,51,51,OWNER,5,junior,18–24


In [103]:
customer_card_age["age_group"].isna().sum()

np.int64(0)

In [104]:
card_type_by_age_group = (
    customer_card_age
    .groupby(
        [
            "age_group",
            "type_card"
        ]
    )
    .agg(
        card_count=("card_id", "count")
    )
    .reset_index()
)

card_type_by_age_group

C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\650762503.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(


,age_group,type_card,card_count
0,Under 18,classic,0
1,Under 18,gold,0
2,Under 18,junior,20
3,18–24,classic,51
4,18–24,gold,5
5,18–24,junior,125
6,25–34,classic,150
7,25–34,gold,20
8,25–34,junior,0
9,35–44,classic,141


In [105]:
card_type_by_age_group["age_group_total"] = (
    card_type_by_age_group
    .groupby("age_group")["card_count"]
    .transform("sum")
)

card_type_by_age_group["percentage_within_age_group"] = (
    card_type_by_age_group["card_count"]
    / card_type_by_age_group["age_group_total"]
    * 100
).round(2)

card_type_by_age_group

C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\2424977570.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("age_group")["card_count"]


,age_group,type_card,card_count,age_group_total,percentage_within_age_group
0,Under 18,classic,0,20,0.00
1,Under 18,gold,0,20,0.00
2,Under 18,junior,20,20,100.00
3,18–24,classic,51,181,28.18
4,18–24,gold,5,181,2.76
5,18–24,junior,125,181,69.06
6,25–34,classic,150,170,88.24
7,25–34,gold,20,170,11.76
8,25–34,junior,0,170,0.00
9,35–44,classic,141,160,88.12


### Interpretation

Card distribution differs noticeably across customer age groups in the dataset.

Among customers under 18, all associated cards are Junior cards. In the 18–24 age group, Junior cards represent 69.06% of associated cards, followed by Classic cards at 28.18% and Gold cards at 2.76%.

For customers aged 25 and above, Classic cards represent the majority of associated cards in every age group, with shares ranging from 80.00% among customers aged 65+ to 88.82% among customers aged 55–64. Gold cards account for approximately 11%–20% across these groups.

These results describe the observed composition of cards associated with customers in different age groups. They do not establish customer preferences, product suitability, profitability, or the reasons for differences between age groups.

The 65+ group should also be interpreted cautiously because it contains only 15 associated cards in this analysis.

In [106]:
customer_loan = (
    disp[
        [
            "client_id",
            "account_id",
            "type"
        ]
    ]
    .merge(
        loan[
            [
                "account_id",
                "loan_id"
            ]
        ],
        on="account_id",
        how="inner"
    )
)

customer_loan.head()

,client_id,account_id,type,loan_id
0,2,2,OWNER,4959
1,3,2,DISPONENT,4959
2,25,19,OWNER,4961
3,31,25,OWNER,4962
4,45,37,OWNER,4967


In [107]:
print("Loan records:", customer_loan["loan_id"].nunique())
print("Customers associated with loans:", customer_loan["client_id"].nunique())

Loan records: 682
Customers associated with loans: 827


In [108]:
customer_loan_owner = (
    customer_loan[
        customer_loan["type"] == "OWNER"
    ]
    .copy()
)

print("Loan records:", customer_loan_owner["loan_id"].nunique())
print("Customers associated with loan accounts:", customer_loan_owner["client_id"].nunique())

Loan records: 682
Customers associated with loan accounts: 682


In [109]:
customer_loan_owner.head()

,client_id,account_id,type,loan_id
0,2,2,OWNER,4959
2,25,19,OWNER,4961
3,31,25,OWNER,4962
4,45,37,OWNER,4967
5,46,38,OWNER,4968


In [110]:
loan_customers = (
    customer_loan_owner[
        ["client_id"]
    ]
    .drop_duplicates()
    .assign(has_loan=True)
)

loan_customers.head()

,client_id,has_loan
0,2,True
2,25,True
3,31,True
4,45,True
5,46,True


In [111]:
card_customers = (
    customer_card[
        customer_card["type_disp"] == "OWNER"
    ][
        ["client_id"]
    ]
    .drop_duplicates()
    .assign(has_card=True)
)

card_customers.head()

,client_id,has_card
0,9,True
1,19,True
2,41,True
3,42,True
4,51,True


In [112]:
customer_product_profile = (
    client[
        [
            "client_id",
            "age_group"
        ]
    ]
    .merge(
        loan_customers,
        on="client_id",
        how="left"
    )
    .merge(
        card_customers,
        on="client_id",
        how="left"
    )
)

customer_product_profile["has_loan"] = (
    customer_product_profile["has_loan"]
    .fillna(False)
)

customer_product_profile["has_card"] = (
    customer_product_profile["has_card"]
    .fillna(False)
)

customer_product_profile.head()

C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\2426123236.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\2426123236.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,client_id,age_group,has_loan,has_card
0,1,25–34,False,False
1,2,45–54,True,False
2,3,55–64,False,False
3,4,35–44,False,False
4,5,35–44,False,False


In [113]:
customer_product_profile["product_profile"] = "Neither"

customer_product_profile.loc[
    customer_product_profile["has_loan"] & ~customer_product_profile["has_card"],
    "product_profile"
] = "Loan Only"

customer_product_profile.loc[
    ~customer_product_profile["has_loan"] & customer_product_profile["has_card"],
    "product_profile"
] = "Card Only"

customer_product_profile.loc[
    customer_product_profile["has_loan"] & customer_product_profile["has_card"],
    "product_profile"
] = "Loan + Card"

In [114]:
customer_product_profile["product_profile"].value_counts()

product_profile
Neither        3965
Card Only       722
Loan Only       512
Loan + Card     170
Name: count, dtype: int64

In [115]:
product_profile_by_age = (
    customer_product_profile
    .groupby(
        [
            "age_group",
            "product_profile"
        ]
    )
    .agg(
        customer_count=("client_id", "count")
    )
    .reset_index()
)

product_profile_by_age

C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\3394623336.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(


,age_group,product_profile,customer_count
0,Under 18,Card Only,20
1,Under 18,Loan + Card,0
2,Under 18,Loan Only,0
3,Under 18,Neither,52
4,18–24,Card Only,149
5,18–24,Loan + Card,32
6,18–24,Loan Only,62
7,18–24,Neither,487
8,25–34,Card Only,131
9,25–34,Loan + Card,39


In [116]:
product_profile_by_age["age_group_total"] = (
    product_profile_by_age
    .groupby("age_group")["customer_count"]
    .transform("sum")
)

product_profile_by_age["percentage_within_age_group"] = (
    product_profile_by_age["customer_count"]
    / product_profile_by_age["age_group_total"]
    * 100
).round(2)

product_profile_by_age

C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\2797906720.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("age_group")["customer_count"]


,age_group,product_profile,customer_count,age_group_total,percentage_within_age_group
0,Under 18,Card Only,20,72,27.78
1,Under 18,Loan + Card,0,72,0.00
2,Under 18,Loan Only,0,72,0.00
3,Under 18,Neither,52,72,72.22
4,18–24,Card Only,149,730,20.41
5,18–24,Loan + Card,32,730,4.38
6,18–24,Loan Only,62,730,8.49
7,18–24,Neither,487,730,66.71
8,25–34,Card Only,131,962,13.62
9,25–34,Loan + Card,39,962,4.05


### Interpretation

Observed financial-product profiles vary across customer age groups in the dataset.

The majority of customers in every age group fall into the "Neither" category, ranging from 66.71% among customers aged 18–24 to 98.17% among customers aged 65+. Among customers aged 25–64, Card Only represents approximately 13%–15% of each age group's population, while Loan Only ranges from 9.85% to 13.31%.

The Loan + Card profile represents a relatively small share across most age groups, with the highest observed proportion among customers aged 45–54 at 4.68%. No owner-associated loan records were observed for customers under 18 or aged 65+ in this analysis.

These results describe observed product associations within the dataset and should not be interpreted as evidence of customer preferences, product suitability, profitability, or the reasons for differences between age groups. The analysis also uses an owner-only attribution rule to avoid assigning the same account-level product to multiple customers.

In [117]:
print("Total customers:", customer_product_profile["client_id"].nunique())
print("Total profile records:", customer_product_profile["product_profile"].count())
print(
    "Profile counts sum:",
    customer_product_profile["product_profile"].value_counts().sum()
)

Total customers: 5369
Total profile records: 5369
Profile counts sum: 5369


In [118]:
product_profile_by_gender = (
    customer_product_profile
    .merge(
        client[
            [
                "client_id",
                "gender"
            ]
        ],
        on="client_id",
        how="left"
    )
    .groupby(
        [
            "gender",
            "product_profile"
        ]
    )
    .agg(
        customer_count=("client_id", "count")
    )
    .reset_index()
)

product_profile_by_gender

,gender,product_profile,customer_count
0,Female,Card Only,338
1,Female,Loan + Card,85
2,Female,Loan Only,263
3,Female,Neither,1959
4,Male,Card Only,384
5,Male,Loan + Card,85
6,Male,Loan Only,249
7,Male,Neither,2006


In [119]:
product_profile_by_gender["gender_total"] = (
    product_profile_by_gender
    .groupby("gender")["customer_count"]
    .transform("sum")
)

product_profile_by_gender["percentage_within_gender"] = (
    product_profile_by_gender["customer_count"]
    / product_profile_by_gender["gender_total"]
    * 100
).round(2)

product_profile_by_gender

,gender,product_profile,customer_count,gender_total,percentage_within_gender
0,Female,Card Only,338,2645,12.78
1,Female,Loan + Card,85,2645,3.21
2,Female,Loan Only,263,2645,9.94
3,Female,Neither,1959,2645,74.06
4,Male,Card Only,384,2724,14.10
5,Male,Loan + Card,85,2724,3.12
6,Male,Loan Only,249,2724,9.14
7,Male,Neither,2006,2724,73.64


### Interpretation

The observed financial-product profiles are broadly similar between the two gender categories in the dataset.

The "Neither" profile represents the largest share for both groups, accounting for 74.06% of female customers and 73.64% of male customers. Card Only represents 12.78% of female customers and 14.10% of male customers, while Loan Only represents 9.94% and 9.14%, respectively.

The Loan + Card profile is almost identical across the two groups, at 3.21% for female customers and 3.12% for male customers.

These results describe observed product associations within the historical dataset and do not establish differences in customer preferences, financial needs, product suitability, profitability, or the causes of the observed variation. Gender is based on the encoding provided in the source data.

In [120]:
customer_district = (
    client[
        [
            "client_id",
            "district_id"
        ]
    ]
    .merge(
        district[
            [
                "district_id",
                "district_name",
                "region"
            ]
        ],
        on="district_id",
        how="left"
    )
)

customer_district.head()

,client_id,district_id,district_name,region
0,1,18,Pisek,south Bohemia
1,2,1,Hl.m. Praha,Prague
2,3,1,Hl.m. Praha,Prague
3,4,5,Kolin,central Bohemia
4,5,5,Kolin,central Bohemia


In [121]:
customer_product_geography = (
    customer_product_profile
    .merge(
        customer_district,
        on="client_id",
        how="left"
    )
)

customer_product_geography.head()

,client_id,age_group,has_loan,has_card,product_profile,district_id,district_name,region
0,1,25–34,False,False,Neither,18,Pisek,south Bohemia
1,2,45–54,True,False,Loan Only,1,Hl.m. Praha,Prague
2,3,55–64,False,False,Neither,1,Hl.m. Praha,Prague
3,4,35–44,False,False,Neither,5,Kolin,central Bohemia
4,5,35–44,False,False,Neither,5,Kolin,central Bohemia


In [122]:
print("Customers:", customer_product_geography["client_id"].nunique())
print("Missing district:", customer_product_geography["district_id"].isna().sum())
print("Missing region:", customer_product_geography["region"].isna().sum())

Customers: 5369
Missing district: 0
Missing region: 0


In [123]:
product_profile_by_region = (
    customer_product_geography
    .groupby(
        [
            "region",
            "product_profile"
        ]
    )
    .agg(
        customer_count=("client_id", "count")
    )
    .reset_index()
)

product_profile_by_region

,region,product_profile,customer_count
0,Prague,Card Only,111
1,Prague,Loan + Card,19
2,Prague,Loan Only,60
3,Prague,Neither,473
4,central Bohemia,Card Only,95
5,central Bohemia,Loan + Card,25
6,central Bohemia,Loan Only,62
7,central Bohemia,Neither,482
8,east Bohemia,Card Only,81
9,east Bohemia,Loan + Card,16


In [124]:
product_profile_by_region["region_total"] = (
    product_profile_by_region
    .groupby("region")["customer_count"]
    .transform("sum")
)

product_profile_by_region["percentage_within_region"] = (
    product_profile_by_region["customer_count"]
    / product_profile_by_region["region_total"]
    * 100
).round(2)

product_profile_by_region

,region,product_profile,customer_count,region_total,percentage_within_region
0,Prague,Card Only,111,663,16.74
1,Prague,Loan + Card,19,663,2.87
2,Prague,Loan Only,60,663,9.05
3,Prague,Neither,473,663,71.34
4,central Bohemia,Card Only,95,664,14.31
5,central Bohemia,Loan + Card,25,664,3.77
6,central Bohemia,Loan Only,62,664,9.34
7,central Bohemia,Neither,482,664,72.59
8,east Bohemia,Card Only,81,660,12.27
9,east Bohemia,Loan + Card,16,660,2.42


### Interpretation

Observed financial-product profiles vary across regions, although the "Neither" profile remains the largest category in every region.

The share of customers in the Card Only profile ranges from 10.65% in north Moravia to 16.74% in Prague. The Loan Only profile ranges from 7.31% in north Bohemia to 11.53% in south Moravia, while the Loan + Card profile remains relatively small across all regions, ranging from 2.42% to 4.01%.

These results indicate descriptive geographic variation in observed product associations. However, the analysis does not establish that region causes differences in product usage, and other customer or account characteristics may contribute to the observed variation.

The results should therefore be treated as exploratory geographic patterns rather than evidence of regional customer preferences or product demand.

In [125]:
top_region_by_product_profile = (
    product_profile_by_region
    .loc[
        product_profile_by_region
        .groupby("product_profile")["percentage_within_region"]
        .idxmax()
    ]
    [
        [
            "product_profile",
            "region",
            "percentage_within_region"
        ]
    ]
    .reset_index(drop=True)
)

top_region_by_product_profile

,product_profile,region,percentage_within_region
0,Card Only,Prague,16.74
1,Loan + Card,south Bohemia,4.01
2,Loan Only,south Moravia,11.53
3,Neither,north Moravia,76.63


In [126]:
order.head()

,order_id,account_id,bank_to,account_to,amount,k_symbol
0,29401,1,YZ,87144583,2452.0,SIPO
1,29402,2,ST,89597016,3372.7,UVER
2,29403,2,QR,13943797,7266.0,SIPO
3,29404,3,WX,83084338,1135.0,SIPO
4,29405,3,CD,24485939,327.0,


In [127]:
order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6471 entries, 0 to 6470
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    6471 non-null   int64  
 1   account_id  6471 non-null   int64  
 2   bank_to     6471 non-null   object 
 3   account_to  6471 non-null   int64  
 4   amount      6471 non-null   float64
 5   k_symbol    6471 non-null   object 
dtypes: float64(1), int64(3), object(2)
memory usage: 303.5+ KB


In [128]:
order["k_symbol"].value_counts(dropna=False)

k_symbol
SIPO        3502
            1379
UVER         717
POJISTNE     532
LEASING      341
Name: count, dtype: int64

### 3.39 Payment Order Distribution

Payment orders are examined by their recorded purpose category to understand the composition of predefined payment activity in the dataset.

The analysis compares both the number and percentage of payment orders associated with each purpose category. Blank or unspecified purpose values are retained as a separate category because their business meaning cannot be determined from the available data.

In [129]:
payment_order_distribution = (
    order["k_symbol"]
    .replace(r"^\s*$", pd.NA, regex=True)
    .fillna("Unspecified")
    .value_counts()
    .reset_index()
)

payment_order_distribution.columns = [
    "payment_order_purpose",
    "order_count"
]

payment_order_distribution["percentage_of_orders"] = (
    payment_order_distribution["order_count"]
    / len(order)
    * 100
).round(2)

payment_order_distribution

,payment_order_purpose,order_count,percentage_of_orders
0,SIPO,3502,54.12
1,Unspecified,1379,21.31
2,UVER,717,11.08
3,POJISTNE,532,8.22
4,LEASING,341,5.27


### 3.40 Payment Order Value by Purpose

Payment orders are compared by their recorded purpose using transaction count, total order value, average order amount, and median order amount.

Comparing both order frequency and monetary value helps distinguish categories with high activity from categories associated with larger individual payment orders.

In [130]:
order_analysis_by_purpose = (
    order.assign(
        payment_order_purpose=(
            order["k_symbol"]
            .replace(r"^\s*$", pd.NA, regex=True)
            .fillna("Unspecified")
        )
    )
    .groupby("payment_order_purpose")
    .agg(
        order_count=("order_id", "count"),
        total_order_value=("amount", "sum"),
        average_order_amount=("amount", "mean"),
        median_order_amount=("amount", "median")
    )
    .reset_index()
)

order_analysis_by_purpose

,payment_order_purpose,order_count,total_order_value,average_order_amount,median_order_amount
0,LEASING,341,759527.1,2227.352199,1951.3
1,POJISTNE,532,686927.0,1291.216165,629.0
2,SIPO,3502,13965417.0,3987.840377,3189.5
3,UVER,717,3035184.5,4233.172245,3990.8
4,Unspecified,1379,2781938.0,2017.358956,1339.0


### Interpretation

SIPO represents the largest payment-order category in the dataset, with 3,502 orders and a total recorded order value of 13,965,417. It therefore accounts for the highest observed payment-order activity by both count and total value.

UVER has substantially fewer orders than SIPO, with 717 orders, but records the highest average order amount at 4,233.17. Its median order amount is also relatively high at 3,990.80.

LEASING has the smallest number of orders among the specified categories, with 341 orders, but its average order amount is higher than that of POJISTNE and Unspecified orders.

These results demonstrate that payment-order frequency and monetary value provide different perspectives on financial activity. A category with fewer orders can still have relatively large individual payment amounts.

The Unspecified category represents 21.31% of all payment orders. Because the purpose of these orders cannot be determined from the available category information, their business interpretation should be treated cautiously.

The analysis is descriptive and does not establish the reasons for differences between payment-order categories.

### 3.41 Payment Orders per Account

Payment-order activity is examined at the account level to understand how widely payment orders are distributed across accounts.

The analysis compares the number of payment orders associated with each account and summarises the distribution of payment-order activity.

In [131]:
orders_per_account = (
    order
    .groupby("account_id")
    .size()
    .reset_index(name="payment_order_count")
)

orders_per_account["payment_order_count"].describe()

count    3758.000000
mean        1.721927
std         0.992172
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         5.000000
Name: payment_order_count, dtype: float64

### 3.42 Distribution of Payment Orders per Account

The distribution of payment orders per account is examined to understand how payment-order activity is distributed across accounts.

The analysis identifies the number of accounts associated with one, two, three, four, or five payment orders.

In [132]:
payment_orders_per_account_distribution = (
    orders_per_account["payment_order_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

payment_orders_per_account_distribution.columns = [
    "payment_order_count",
    "account_count"
]

payment_orders_per_account_distribution["percentage_of_accounts"] = (
    payment_orders_per_account_distribution["account_count"]
    / len(orders_per_account)
    * 100
).round(2)

payment_orders_per_account_distribution

,payment_order_count,account_count,percentage_of_accounts
0,1,2103,55.96
1,2,949,25.25
2,3,416,11.07
3,4,228,6.07
4,5,62,1.65


### Interpretation

Payment-order activity is concentrated among accounts with a relatively small number of recorded orders.

Among the 3,758 accounts with at least one payment order, 55.96% have exactly one payment order and 25.25% have two payment orders. Combined, 81.21% of accounts have one or two payment orders.

Only 7.72% of accounts have four or five payment orders, while no account in the dataset has more than five recorded payment orders.

These results describe the distribution of recorded payment orders across accounts. They do not indicate why some accounts have more payment orders or whether a higher number of orders represents greater customer engagement or financial importance.

### 3.43 Payment Order Coverage Across Accounts

Payment-order coverage is examined across all accounts to determine the proportion of accounts with at least one recorded payment order.

Unlike the previous analysis, this calculation includes accounts with zero payment orders.

In [133]:
account_payment_order_summary = (
    account[["account_id"]]
    .merge(
        orders_per_account,
        on="account_id",
        how="left"
    )
)

account_payment_order_summary["payment_order_count"] = (
    account_payment_order_summary["payment_order_count"]
    .fillna(0)
    .astype(int)
)

account_payment_order_summary["has_payment_order"] = (
    account_payment_order_summary["payment_order_count"] > 0
)

account_payment_order_summary["has_payment_order"].value_counts()

has_payment_order
True     3758
False     742
Name: count, dtype: int64

In [134]:
payment_order_coverage = (
    account_payment_order_summary["has_payment_order"]
    .mean()
    * 100
)

payment_order_coverage

np.float64(83.51111111111112)

### Interpretation

Of the 4,500 accounts in the dataset, 3,758 accounts (83.51%) have at least one recorded payment order, while 742 accounts (16.49%) have none.

This indicates that payment orders are associated with a large proportion of accounts in the dataset. However, the presence of a payment order does not by itself indicate customer engagement, profitability, or financial importance.

The analysis is descriptive and reflects the recorded payment-order relationships available in the dataset.

In [135]:
id="h5k9p2"
print("Total accounts:", len(account_payment_order_summary))
print(
    "Accounts with payment orders:",
    account_payment_order_summary["has_payment_order"].sum()
)
print(
    "Accounts without payment orders:",
    (~account_payment_order_summary["has_payment_order"]).sum()
)

Total accounts: 4500
Accounts with payment orders: 3758
Accounts without payment orders: 742


### 3.44 Negative Recorded Balances

Negative recorded balances are examined as a potential financial risk indicator.

The analysis identifies how frequently negative balances occur and examines their distribution across accounts. A negative balance is treated as an observed account condition rather than evidence of fraud or financial distress.

In [136]:
negative_balance_transactions = trans[
    trans["balance"] < 0
].copy()

print(
    "Transactions with negative balances:",
    len(negative_balance_transactions)
)

print(
    "Percentage of transactions with negative balances:",
    round(
        len(negative_balance_transactions) / len(trans) * 100,
        2
    )
)

print(
    "Accounts with negative balances:",
    negative_balance_transactions["account_id"].nunique()
)

Transactions with negative balances: 2999
Percentage of transactions with negative balances: 0.28
Accounts with negative balances: 288


### Interpretation

Negative recorded balances are relatively uncommon at the transaction level. Of the 1,056,320 recorded transactions, 2,999 transactions (0.28%) have a negative recorded balance.

These transactions are associated with 288 distinct accounts. This indicates that negative balances are rare across overall transaction activity but occur across multiple accounts rather than being concentrated in a single account.

A negative recorded balance is treated as a potential financial risk indicator rather than evidence of financial distress, default, or fraud. Additional account-level and transaction-level information would be required to determine the underlying causes and significance of these observations.

### 3.45 Negative Balance Frequency per Account

Negative-balance transactions are examined at the account level to determine whether accounts with negative recorded balances experienced the condition once or repeatedly.

The analysis helps distinguish isolated observations from accounts with multiple negative-balance transactions.

In [137]:
negative_balance_by_account = (
    negative_balance_transactions
    .groupby("account_id")
    .size()
    .reset_index(name="negative_balance_transaction_count")
)

negative_balance_by_account[
    "negative_balance_transaction_count"
].describe()

count    288.000000
mean      10.413194
std       15.807182
min        1.000000
25%        2.000000
50%        6.000000
75%       11.000000
max      124.000000
Name: negative_balance_transaction_count, dtype: float64

### Interpretation

Among the 288 accounts with at least one negative recorded balance, the median number of negative-balance transactions is 6, while the mean is 10.41.

This indicates that negative recorded balances were not exclusively isolated observations. At least half of the affected accounts had six or more transactions with a negative recorded balance.

The distribution also shows substantial variation between accounts. The maximum observed value is 124 negative-balance transactions for a single account, while the middle 50% of accounts have between 2 and 11 such transactions.

These results identify repeated negative-balance observations that may warrant further investigation. However, frequency of negative balances alone is not sufficient to classify an account as high risk or financially distressed.

### 3.46 Accounts with Repeated Negative Balances

Accounts with the highest number of negative-balance transactions are identified as potential investigation candidates.

The purpose of this analysis is to prioritize unusual account-level patterns for further review, not to classify accounts as fraudulent or financially distressed.

In [138]:
top_negative_balance_accounts = (
    negative_balance_by_account
    .sort_values(
        "negative_balance_transaction_count",
        ascending=False
    )
    .head(10)
)

top_negative_balance_accounts

,account_id,negative_balance_transaction_count
68,2335,124
101,3678,111
64,2260,100
67,2305,73
97,3374,70
152,5343,63
46,1539,61
160,5429,52
144,5092,52
123,4356,48


### Interpretation

The accounts with the highest number of negative-balance transactions show repeated occurrences rather than isolated observations.

Account 2335 has the highest observed count, with 124 transactions recorded with a negative balance, followed by account 3678 with 111 and account 2260 with 100.

The remaining accounts in the top 10 have between 48 and 73 negative-balance transactions.

These accounts can be considered potential candidates for further investigation because of the repeated negative-balance observations. However, the frequency of negative balances alone is not sufficient to classify an account as high risk, financially distressed, or fraudulent.

Further analysis should compare negative-balance frequency with overall transaction activity and transaction value to determine whether these observations are relatively unusual for each account.

### 3.47 Negative Balance Rate by Account

The negative-balance rate is calculated to measure the proportion of each account's transactions that have a negative recorded balance.

This provides additional context to the absolute number of negative-balance transactions, since accounts may have very different levels of overall transaction activity.

In [139]:
negative_balance_rate = (
    negative_balance_by_account
    .merge(
        transactions_per_account,
        on="account_id",
        how="left"
    )
)

negative_balance_rate["negative_balance_rate"] = (
    negative_balance_rate["negative_balance_transaction_count"]
    / negative_balance_rate["transaction_count"]
    * 100
)

negative_balance_rate = (
    negative_balance_rate
    .sort_values(
        "negative_balance_rate",
        ascending=False
    )
)

negative_balance_rate.head(10)

,account_id,negative_balance_transaction_count,transaction_count,negative_balance_rate
68,2335,124,329,37.689970
64,2260,100,281,35.587189
160,5429,52,156,33.333333
127,4588,39,126,30.952381
267,10343,31,112,27.678571
101,3678,111,414,26.811594
67,2305,73,299,24.414716
117,4127,33,142,23.239437
217,7520,40,175,22.857143
152,5343,63,298,21.140940


### Interpretation

The negative-balance rate provides additional context by measuring the proportion of an account's transactions associated with a negative recorded balance.

Account 2335 has the highest negative-balance rate among the accounts shown, with 37.69% of its 329 recorded transactions associated with a negative balance. Account 2260 follows with 35.59%, while account 5429 has a rate of 33.33%.

The results also demonstrate why relative frequency can provide a different perspective from absolute counts. Some accounts with fewer negative-balance transactions can have a relatively high negative-balance rate because they also have lower overall transaction activity.

These rates should be interpreted together with transaction volume and other account-level indicators. A high negative-balance rate alone does not establish financial distress, default, or fraud. The results are best treated as anomaly or risk indicators that may help prioritize accounts for further investigation.

### 3.48 Negative Balance Frequency and Rate

Negative-balance frequency and negative-balance rate are examined together to identify accounts with repeated negative-balance observations that are also relatively frequent within their overall transaction activity.

Using both measures provides additional context and reduces the risk of interpreting a single metric in isolation.

In [140]:
negative_balance_risk_view = (
    negative_balance_rate[
        [
            "account_id",
            "negative_balance_transaction_count",
            "transaction_count",
            "negative_balance_rate"
        ]
    ]
    .copy()
)

negative_balance_risk_view = (
    negative_balance_risk_view
    .sort_values(
        [
            "negative_balance_rate",
            "negative_balance_transaction_count"
        ],
        ascending=[False, False]
    )
)

negative_balance_risk_view.head(15)

,account_id,negative_balance_transaction_count,transaction_count,negative_balance_rate
68,2335,124,329,37.689970
64,2260,100,281,35.587189
160,5429,52,156,33.333333
127,4588,39,126,30.952381
267,10343,31,112,27.678571
101,3678,111,414,26.811594
67,2305,73,299,24.414716
117,4127,33,142,23.239437
217,7520,40,175,22.857143
152,5343,63,298,21.140940


### Interpretation

Examining negative-balance frequency together with negative-balance rate provides a more complete view of the observed account-level pattern.

Account 2335 stands out with both a high number of negative-balance transactions (124) and a high negative-balance rate (37.69%). Account 2260 also shows a high frequency and rate, with 100 negative-balance transactions representing 35.59% of its recorded transactions.

Other accounts illustrate why both measures are useful. For example, account 3711 has a relatively high negative-balance rate of 20.79%, but only 21 negative-balance transactions across 101 total transactions.

These measures should therefore be considered together when identifying accounts for further investigation. They remain descriptive indicators and do not constitute a formal risk score or evidence of financial distress, default, or fraud.

### 3.49 Large Transaction Amounts

Large transaction amounts are examined as a potential transaction-level anomaly indicator.

The analysis identifies transactions with unusually high recorded amounts and examines the accounts associated with them. Large transaction amounts are not treated as evidence of fraud or financial misconduct.

In [141]:
large_transactions = (
    trans[
        [
            "trans_id",
            "account_id",
            "transaction_date",
            "transaction_type_english",
            "amount",
            "balance"
        ]
    ]
    .sort_values(
        "amount",
        ascending=False
    )
    .head(20)
)

large_transactions

,trans_id,account_id,transaction_date,transaction_type_english,amount,balance
1047413,291346,998,1998-12-15,Withdrawal,87400.0,22168.9
880698,1306770,4462,1998-06-06,Withdrawal,87300.0,14321.1
335550,855512,2915,1996-06-14,Withdrawal,86400.0,29421.5
892659,1207200,4130,1998-06-19,Withdrawal,85800.0,5051.6
891095,1170721,4014,1998-06-15,Withdrawal,81900.0,21885.1
338463,1248236,4269,1996-06-28,Withdrawal,80700.0,19715.5
576138,1137293,3886,1997-06-14,Withdrawal,80700.0,6583.5
174510,2648606,8759,1995-06-07,Withdrawal,80400.0,7359.2
578967,115552,390,1997-06-22,Withdrawal,80400.0,23567.1
885035,706371,2419,1998-06-10,Withdrawal,80100.0,31394.0


### Interpretation

The largest recorded transactions are all classified as Withdrawals, with amounts ranging from 77,700 to 87,400 among the 20 transactions examined.

Several of the largest transactions occurred during June, including multiple observations in 1995, 1996, 1997, and 1998. This indicates a concentration of high-value recorded transactions within the Withdrawal category and, within the observed sample, a notable concentration during June.

Two of the transactions in the top 20 also resulted in negative recorded balances. For example, a transaction of 78,000 resulted in a recorded balance of -11.0, while a transaction of 77,700 resulted in a balance of -929.1.

These observations may warrant further investigation because they combine two potentially relevant indicators: unusually large transaction amounts and negative recorded balances. However, a large transaction or a negative balance alone does not establish fraud, financial distress, or other misconduct. Additional contextual information would be required to determine the underlying cause of these observations.

### 3.50 Large Transactions with Negative Balances

Transactions with both relatively large recorded amounts and negative balances are examined as combined anomaly indicators.

The purpose is to identify transaction-level observations that may warrant further investigation based on multiple characteristics rather than a single metric.

In [142]:
large_transaction_threshold = trans["amount"].quantile(0.99)

large_transaction_threshold

np.float64(45454.80999999994)

In [143]:
large_negative_transactions = (
    trans[
        (trans["amount"] >= large_transaction_threshold)
        & (trans["balance"] < 0)
    ]
    [
        [
            "trans_id",
            "account_id",
            "transaction_date",
            "transaction_type_english",
            "amount",
            "balance"
        ]
    ]
    .sort_values(
        "amount",
        ascending=False
    )
)

large_negative_transactions.head(20)

,trans_id,account_id,transaction_date,transaction_type_english,amount,balance
565641,3183668,10562,1997-06-03,Withdrawal,78000.0,-11.0
335726,1669417,5661,1996-06-14,Withdrawal,77700.0,-929.1
892629,2471390,8153,1998-06-19,Withdrawal,61800.0,-313.2
175090,868869,2959,1995-06-08,Withdrawal,59700.0,-969.1
66667,1959066,6636,1994-06-14,Withdrawal,58200.0,-1059.3
67853,102039,347,1994-06-26,Withdrawal,51900.0,-243.3
328564,75951,253,1996-06-03,Withdrawal,47680.0,-1212.9
67994,2407931,7944,1994-06-28,Withdrawal,45760.0,-1918.1


### Interpretation

Using the 99th percentile of transaction amounts as the threshold for unusually large transactions, the threshold was 45,454.81.

Eight transactions satisfied both conditions: a transaction amount at or above this threshold and a negative recorded balance.

All eight transactions were classified as Withdrawals. The largest observation was a withdrawal of 78,000 associated with account 10562, which resulted in a recorded balance of -11.0. The second-largest observation was a withdrawal of 77,700 associated with account 5661, followed by a negative recorded balance of -929.1.

All eight observations occurred during June. This concentration is notable because several of the largest transactions identified earlier also occurred during June.

These observations may warrant further investigation because they combine two potentially relevant indicators: unusually large transaction amounts and negative recorded balances. However, they do not provide evidence of fraud or other misconduct. The concentration in June also does not establish that June represents a higher-risk period, as the underlying cause cannot be determined from the available data.

### 3.51 Accounts with Multiple Anomaly Indicators

Account-level anomaly indicators are combined to identify accounts that exhibit more than one potentially unusual transaction characteristic.

The analysis identifies accounts with both negative recorded balances and transactions in the top 1% of transaction amounts.

The resulting accounts are treated as potential candidates for further investigation rather than as confirmed high-risk or fraudulent accounts.

In [144]:
accounts_with_negative_balance = set(
    negative_balance_transactions["account_id"]
)

accounts_with_large_transactions = set(
    trans.loc[
        trans["amount"] >= large_transaction_threshold,
        "account_id"
    ]
)

accounts_with_multiple_indicators = (
    accounts_with_negative_balance
    & accounts_with_large_transactions
)

print(
    "Accounts with both indicators:",
    len(accounts_with_multiple_indicators)
)

Accounts with both indicators: 197


In [145]:
multiple_indicator_accounts = (
    negative_balance_rate[
        negative_balance_rate["account_id"].isin(
            accounts_with_multiple_indicators
        )
    ]
    .sort_values(
        [
            "negative_balance_rate",
            "negative_balance_transaction_count"
        ],
        ascending=[False, False]
    )
)

multiple_indicator_accounts

,account_id,negative_balance_transaction_count,transaction_count,negative_balance_rate
103,3711,21,101,20.792079
130,4618,29,149,19.463087
97,3374,70,369,18.970190
30,867,14,83,16.867470
29,849,21,136,15.441176
...,...,...,...,...
159,5428,1,458,0.218341
229,8212,1,459,0.217865
274,10670,1,468,0.213675
280,11021,1,499,0.200401


### Interpretation

A total of 197 accounts have both observed indicators: at least one negative recorded balance and at least one transaction within the top 1% of transaction amounts.

The frequency of negative-balance observations varies substantially across these accounts. For example, account 3711 has 21 negative-balance transactions across 101 total transactions, corresponding to a rate of 20.79%, while account 11295 has only one negative-balance transaction across 563 total transactions, corresponding to a rate of 0.18%.

This demonstrates that the presence of multiple indicators does not imply the same level or intensity of unusual activity across accounts.

The 197 accounts should therefore be treated as a population of accounts exhibiting multiple observed indicators that may warrant further investigation, rather than as confirmed high-risk or fraudulent accounts.

# 4. Statistical Analysis

This section applies statistical techniques to better understand relationships, distributions, and differences within the financial services data.

The analysis focuses on descriptive statistics and relationships between variables rather than predictive modeling.

### 4.1 Transaction Amount Distribution

The distribution of transaction amounts is examined to understand the typical transaction size and the presence of extreme values.

Mean, median, and percentile statistics are compared because transaction amounts may be unevenly distributed and influenced by high-value transactions.

In [146]:
transaction_amount_statistics = (
    trans["amount"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .to_frame(name="transaction_amount")
)

transaction_amount_statistics

,transaction_amount
count,1.056320e+06
mean,5.924146e+03
std,9.522735e+03
min,0.000000e+00
1%,1.460000e+01
5%,1.460000e+01
25%,1.359000e+02
50%,2.100000e+03
75%,6.800000e+03
95%,2.570000e+04


In [147]:
mean_transaction_amount = trans["amount"].mean()
median_transaction_amount = trans["amount"].median()

mean_median_difference = (
    mean_transaction_amount - median_transaction_amount
)

print("Mean transaction amount:", round(mean_transaction_amount, 2))
print("Median transaction amount:", round(median_transaction_amount, 2))
print("Mean - median:", round(mean_median_difference, 2))

Mean transaction amount: 5924.15
Median transaction amount: 2100.0
Mean - median: 3824.15


### Interpretation

The mean transaction amount is 5,924.15, while the median is 2,100.00. The substantial difference between these measures indicates that the transaction amount distribution is positively skewed.

The presence of relatively large transactions increases the mean, while the median remains less affected by extreme observations. Therefore, the median provides a more representative measure of a typical transaction amount in this dataset.

For transaction analysis, mean and median should be considered together because relying only on the mean could overstate the typical transaction size.

In [148]:
transaction_percentiles = (
    trans["amount"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .reset_index()
)

transaction_percentiles.columns = [
    "percentile",
    "transaction_amount"
]

transaction_percentiles

,percentile,transaction_amount
0,0.50,2100.00
1,0.75,6800.00
2,0.90,18400.00
3,0.95,25700.00
4,0.99,45454.81


### Interpretation

The percentile distribution shows that transaction amounts are concentrated toward the lower end of the distribution, with a relatively small number of substantially larger transactions.

The median transaction amount is 2,100, while the 75th percentile is 6,800. This means that 75% of recorded transactions have an amount of 6,800 or less.

The 90th percentile is 18,400, while the 99th percentile is 45,454.81. The large increase between these percentile levels indicates that the upper tail of the distribution contains relatively large transactions.

These results support the earlier observation that transaction amounts are positively skewed and that the mean should be interpreted together with the median and percentile statistics.

### 4.3 Correlation Between Account Activity and Transaction Value

The relationship between account transaction frequency and total transaction value is examined using the Pearson correlation coefficient.

The analysis evaluates whether accounts with more recorded transactions also tend to have higher total transaction values.

Correlation measures the strength and direction of a linear relationship between two numerical variables. It does not establish causation.

In [149]:
account_activity_correlation = (
    account_activity_value[
        [
            "transaction_count",
            "total_transaction_value"
        ]
    ]
    .corr()
)

account_activity_correlation

,transaction_count,total_transaction_value
transaction_count,1.000000,0.610739
total_transaction_value,0.610739,1.000000


In [150]:
correlation_transaction_count_value = (
    account_activity_value[
        "transaction_count"
    ]
    .corr(
        account_activity_value[
            "total_transaction_value"
        ]
    )
)

print(
    "Pearson correlation:",
    round(correlation_transaction_count_value, 3)
)

Pearson correlation: 0.611


### 4.4 Transaction Amount by Transaction Type

Transaction amounts are compared across transaction types to determine whether different types of financial activity have different typical transaction sizes.

Mean, median, and percentile statistics are compared because transaction amounts are positively skewed and may contain relatively large observations.

In [151]:
transaction_amount_by_type = (
    trans
    .groupby("transaction_type_english")["amount"]
    .agg(
        transaction_count="count",
        mean_amount="mean",
        median_amount="median",
        q75=lambda x: x.quantile(0.75),
        q90=lambda x: x.quantile(0.90),
        q99=lambda x: x.quantile(0.99)
    )
    .reset_index()
)

transaction_amount_by_type

,transaction_type_english,transaction_count,mean_amount,median_amount,q75,q90,q99
0,Cash Withdrawal,16666,12516.726149,12103.0,18081.0,22020.5,24652.0
1,Credit,405083,7967.434882,1600.0,12087.5,24400.0,47883.0
2,Withdrawal,634571,4446.653531,2042.0,5200.0,11601.0,37700.0


### Interpretation

Transaction amount distributions differ substantially across transaction types.

Cash Withdrawal has the highest mean and median transaction amounts, at 12,516.73 and 12,103.00 respectively. The relatively small difference between these measures indicates that the distribution is less affected by extreme values than the other transaction types.

Credit transactions show the largest difference between the mean and median. The mean is 7,967.43, while the median is only 1,600.00. This indicates a strongly positively skewed distribution in which relatively large Credit transactions have a substantial effect on the mean.

Withdrawal transactions also show positive skewness, with a mean of 4,446.65 compared with a median of 2,042.00.

These results demonstrate why transaction analysis should consider both mean and median rather than relying on a single measure of central tendency.

### 4.5 Statistical Comparison of Transaction Amounts

The Kruskal–Wallis test is used to assess whether transaction amounts differ statistically across the three transaction types.

This non-parametric test is appropriate for the analysis because transaction amounts are strongly skewed and contain relatively large observations.

The test evaluates whether the distributions of transaction amounts differ across Credit, Withdrawal, and Cash Withdrawal transactions.

A statistically significant result indicates that at least one group differs from another, but it does not identify which specific groups differ. Statistical significance also does not imply that the difference is practically or commercially important.

In [152]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [153]:
from scipy.stats import kruskal

In [154]:
cash_withdrawal = trans.loc[
    trans["transaction_type_english"] == "Cash Withdrawal",
    "amount"
]

credit = trans.loc[
    trans["transaction_type_english"] == "Credit",
    "amount"
]

withdrawal = trans.loc[
    trans["transaction_type_english"] == "Withdrawal",
    "amount"
]

kruskal_result = kruskal(
    cash_withdrawal,
    credit,
    withdrawal
)

print("Kruskal-Wallis H-statistic:", round(kruskal_result.statistic, 3))
print("p-value:", kruskal_result.pvalue)

Kruskal-Wallis H-statistic: 31591.155
p-value: 0.0


### Interpretation

The Kruskal–Wallis test produced an H-statistic of 31,591.155 and a p-value below the reporting precision of the calculation.

Using a significance level of 0.05, the result provides strong statistical evidence that transaction amount distributions differ across the three transaction types.

However, the Kruskal–Wallis test does not identify which specific transaction types differ from each other. A post-hoc pairwise analysis is therefore required to examine the differences between individual transaction-type pairs.

Statistical significance should also be interpreted separately from practical or business significance. The large sample size means that relatively small differences may also produce statistically significant results.

In [155]:
%pip show scikit-posthocs

Name: scikit-posthocs
Version: 0.17.0
Summary: Statistical post-hoc analysis and outlier detection algorithms
Home-page: https://github.com/maximtrp/scikit-posthocs
Author: 
Author-email: Maksim Terpilovskii <maximtrp@gmail.com>
License: Copyright (c) 2026 Maksim Terpilovskii

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR 

In [156]:
%pip install scikit-posthocs

Note: you may need to restart the kernel to use updated packages.


In [157]:
import scikit_posthocs as sp

### 4.6 Post-hoc Pairwise Analysis

Following the statistically significant Kruskal–Wallis test, a post-hoc Dunn's test is performed to identify which pairs of transaction types have statistically different transaction amount distributions.

A Bonferroni correction is applied to account for multiple pairwise comparisons.

The analysis compares:
- Cash Withdrawal vs Credit
- Cash Withdrawal vs Withdrawal
- Credit vs Withdrawal

The results indicate statistical differences between groups but do not, by themselves, establish practical or business significance.

In [158]:
dunn_results = sp.posthoc_dunn(
    trans,
    val_col="amount",
    group_col="transaction_type_english",
    p_adjust="bonferroni"
)

dunn_results

,Cash Withdrawal,Credit,Withdrawal
Cash Withdrawal,1.0,0.0,0.0
Credit,0.0,1.0,0.0
Withdrawal,0.0,0.0,1.0


### Interpretation

The Dunn's post-hoc test with Bonferroni correction produced adjusted p-values below the reporting precision of the calculation for all three pairwise comparisons.

This provides statistical evidence that transaction amount distributions differ between Cash Withdrawal and Credit, Cash Withdrawal and Withdrawal, and Credit and Withdrawal.

The results are consistent with the descriptive statistics, which showed substantial differences in the mean and median transaction amounts across transaction types.

However, statistical significance does not indicate the magnitude or business importance of the differences. The very large number of observations in the dataset can make relatively small differences statistically significant. Therefore, the statistical results should be interpreted together with descriptive measures such as the mean, median, and percentiles.

### 4.7 Correlation Between Transaction Count and Average Transaction Amount

The relationship between transaction frequency and average transaction amount is examined at the account level.

The analysis evaluates whether accounts with more recorded transactions also tend to have higher or lower average transaction amounts.

Pearson correlation is used to measure the strength and direction of the linear relationship between the two numerical variables. Correlation does not establish causation.

In [159]:
account_frequency_amount_correlation = (
    account_activity_value[
        [
            "transaction_count",
            "average_transaction_value"
        ]
    ].corr()
)

account_frequency_amount_correlation

,transaction_count,average_transaction_value
transaction_count,1.000000,0.076441
average_transaction_value,0.076441,1.000000


In [160]:
correlation_frequency_average_amount = (
    account_activity_value["transaction_count"].corr(
        account_activity_value["average_transaction_value"]
    )
)

print(
    "Pearson correlation:",
    round(correlation_frequency_average_amount, 3)
)

Pearson correlation: 0.076


### Interpretation

The Pearson correlation between transaction count and average transaction amount is 0.076, indicating a very weak positive linear relationship.

This suggests that transaction frequency has little linear association with the typical transaction size at the account level. Accounts with more transactions do not necessarily process larger individual transactions on average.

This result contrasts with the moderate positive correlation previously observed between transaction count and total transaction value (r = 0.611). Therefore, accounts with more transactions tend to generate higher cumulative transaction value, but this appears to be driven more by the number of transactions than by a strong increase in the average transaction amount.

Correlation measures association and does not establish causation.

### 4.8 Correlation Between Transaction Count and Negative-Balance Rate

The relationship between account transaction frequency and the proportion of transactions associated with negative recorded balances is examined.

The analysis evaluates whether accounts with more recorded transactions tend to have a higher negative-balance rate.

Only accounts with at least one negative recorded balance are included in this analysis because the negative-balance rate is defined for accounts where this condition has been observed.

Pearson correlation is used to measure the strength and direction of the linear relationship. Correlation does not establish causation.

In [161]:
negative_balance_frequency_correlation = (
    negative_balance_rate[
        [
            "transaction_count",
            "negative_balance_rate"
        ]
    ].corr()
)

negative_balance_frequency_correlation

,transaction_count,negative_balance_rate
transaction_count,1.000000,-0.200916
negative_balance_rate,-0.200916,1.000000


In [162]:
correlation_transaction_count_negative_rate = (
    negative_balance_rate["transaction_count"].corr(
        negative_balance_rate["negative_balance_rate"]
    )
)

print(
    "Pearson correlation:",
    round(correlation_transaction_count_negative_rate, 3)
)

Pearson correlation: -0.201


### Interpretation

The Pearson correlation between transaction count and negative-balance rate is -0.201, indicating a weak negative linear relationship.

This suggests that accounts with more recorded transactions tend to have slightly lower negative-balance rates within the group of accounts that experienced at least one negative recorded balance. However, the relationship is weak, indicating that transaction frequency alone does not strongly explain differences in negative-balance rates.

The result does not establish causation. Other account characteristics or transaction patterns may contribute to differences in negative-balance frequency.

This analysis should therefore be interpreted as a descriptive relationship rather than evidence that higher transaction activity reduces financial risk.

### 4.9 Effect Size of Transaction-Type Differences

The statistical significance of differences in transaction amounts does not indicate how large or practically important those differences are.

To complement the Kruskal–Wallis test, epsilon-squared (ε²) is calculated as an effect-size measure.

Effect size provides additional context by estimating the proportion of variability in the ranked transaction amounts that is associated with differences between transaction types.

A small effect size would indicate that, although the differences are statistically significant, transaction type explains only a limited portion of the variation in transaction amounts.

Effect size should therefore be interpreted together with the descriptive statistics and hypothesis-test results.

In [163]:
epsilon_squared = (
    kruskal_result.statistic
    - 3
) / (
    len(trans) - 3
)

print(
    "Kruskal-Wallis epsilon-squared:",
    round(epsilon_squared, 4)
)

Kruskal-Wallis epsilon-squared: 0.0299


### Interpretation

The Kruskal–Wallis epsilon-squared effect size is 0.0299.

This indicates that differences between transaction types account for a relatively small proportion of the variation in ranked transaction amounts.

The result complements the statistically significant Kruskal–Wallis test. Although transaction amount distributions differ statistically across Credit, Withdrawal, and Cash Withdrawal transactions, the effect size indicates that transaction type alone does not explain a large proportion of the overall variation.

This distinction is important because the dataset contains more than one million transactions. With such a large sample, relatively modest differences can produce very small p-values.

Therefore, both statistical significance and effect size should be considered when evaluating whether a finding is meaningful from a business perspective.

### 4.10 Correlation Matrix of Key Account Metrics

A correlation matrix is used to examine the relationships between key account-level activity and risk-indicator metrics.

The analysis compares transaction frequency, total transaction value, average transaction value, and negative-balance rate.

Pearson correlation coefficients are used to measure the strength and direction of linear relationships between the variables.

The results are descriptive and do not establish causation.

In [167]:
account_correlation_data = (
    account_activity_value[
        [
            "account_id",
            "transaction_count",
            "total_transaction_value",
            "average_transaction_value"
        ]
    ]
    .merge(
        negative_balance_rate[
            [
                "account_id",
                "negative_balance_rate"
            ]
        ],
        on="account_id",
        how="left"
    )
)

account_correlation_matrix = (
    account_correlation_data[
        [
            "transaction_count",
            "total_transaction_value",
            "average_transaction_value",
            "negative_balance_rate"
        ]
    ]
    .corr()
)

account_correlation_matrix

,transaction_count,total_transaction_value,average_transaction_value,negative_balance_rate
transaction_count,1.000000,0.610739,0.076441,-0.200916
total_transaction_value,0.610739,1.000000,0.745539,-0.342922
average_transaction_value,0.076441,0.745539,1.000000,-0.330423
negative_balance_rate,-0.200916,-0.342922,-0.330423,1.000000


In [165]:
negative_balance_rate.columns

Index(['account_id', 'negative_balance_transaction_count', 'transaction_count',
       'negative_balance_rate'],
      dtype='object')

In [166]:
account_activity_value.columns

Index(['account_id', 'transaction_count', 'total_transaction_value',
       'average_transaction_value'],
      dtype='object')

### Interpretation

The correlation matrix highlights several relationships between account activity, transaction value, and negative-balance indicators.

Transaction count has a moderate positive correlation with total transaction value (r = 0.611), while its correlation with average transaction value is very weak (r = 0.076). This supports the earlier finding that differences in total transaction value are more strongly associated with transaction frequency than with a consistent increase in average transaction size.

Total transaction value has a relatively strong positive correlation with average transaction value (r = 0.746). This relationship should be interpreted cautiously because total transaction value is mathematically determined by transaction count and average transaction value.

Negative-balance rate has weak negative correlations with transaction count (r = -0.201), total transaction value (r = -0.343), and average transaction value (r = -0.330). These results indicate that accounts with higher transaction values tend to have somewhat lower negative-balance rates within the analysed population. However, these are descriptive associations and do not establish causation.

Overall, the correlation matrix demonstrates that different dimensions of account activity capture different aspects of customer and account behaviour and should not be treated as interchangeable measures.

### 4.11 Transaction Activity Across Customer Age Groups

Transaction activity is compared across customer age groups to examine whether account transaction frequency differs between groups.

The analysis uses owner-associated accounts to avoid attributing the same shared-account transactions to multiple customers.

A Kruskal–Wallis test is used because there are multiple independent age groups and transaction counts may not follow a normal distribution.

A statistically significant result would indicate that at least one age group differs from another in transaction activity. The test does not identify which specific groups differ.

Statistical significance does not necessarily indicate that the differences are practically or commercially important.

In [168]:
age_group_transaction_activity = (
    owner_accounts
    .merge(
        client[
            [
                "client_id",
                "age_group"
            ]
        ],
        on="client_id",
        how="left"
    )
    .merge(
        transactions_per_account,
        on="account_id",
        how="left"
    )
)

age_group_transaction_activity.head()

,client_id,account_id,age_group,transaction_count
0,1,1,25–34,239
1,2,2,45–54,478
2,4,3,35–44,117
3,6,4,65+,186
4,7,5,65+,84


In [169]:
age_group_transaction_activity.groupby(
    "age_group"
)["transaction_count"].describe()

C:\Users\alexa\AppData\Local\Temp\ipykernel_23908\1217982030.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_group_transaction_activity.groupby(


,count,mean,std,min,25%,50%,75%,max
age_group,,,,,,,,
Under 18,57.0,223.754386,118.333862,29.0,130.0,209.0,316.00,433.0
18–24,658.0,230.214286,129.125007,12.0,130.0,205.5,322.50,675.0
25–34,819.0,242.627595,131.680772,9.0,143.0,217.0,335.00,661.0
35–44,798.0,243.392231,131.171989,11.0,140.0,213.0,348.00,628.0
45–54,832.0,247.090144,129.323837,11.0,141.5,221.0,352.25,637.0
55–64,697.0,233.562410,128.047829,13.0,130.0,205.0,326.00,665.0
65+,639.0,204.654147,101.781964,41.0,118.0,182.0,304.00,434.0


In [170]:
age_group_samples = [
    group["transaction_count"].values
    for _, group in age_group_transaction_activity.groupby(
        "age_group",
        observed=True
    )
]

age_group_kruskal_result = kruskal(
    *age_group_samples
)

print(
    "Kruskal-Wallis H-statistic:",
    round(age_group_kruskal_result.statistic, 3)
)

print(
    "p-value:",
    age_group_kruskal_result.pvalue
)

Kruskal-Wallis H-statistic: 45.068
p-value: 4.536545215092608e-08


### Interpretation

The Kruskal–Wallis test produced an H-statistic of 45.068 and a p-value of 4.54 × 10⁻⁸.

Using a significance level of 0.05, the result provides strong statistical evidence that transaction activity differs across at least one pair of customer age groups.

Descriptively, the 45–54 age group has the highest average transaction count at 247.09 transactions per account, while the 65+ age group has the lowest average at 204.65.

However, the Kruskal–Wallis test does not identify which specific age groups differ from each other. A post-hoc pairwise analysis is therefore required.

The observed differences should also be interpreted carefully. Statistical significance does not necessarily imply that the differences are practically or commercially important, particularly given the size of the dataset.

The analysis is descriptive and does not establish that age causes differences in transaction activity.

In [171]:
age_group_dunn_results = sp.posthoc_dunn(
    age_group_transaction_activity,
    val_col="transaction_count",
    group_col="age_group",
    p_adjust="bonferroni"
)

age_group_dunn_results

,Under 18,18–24,25–34,35–44,45–54,55–64,65+
Under 18,1.0,1.000000,1.000000,1.000000,1.000000e+00,1.000000,1.000000e+00
18–24,1.0,1.000000,1.000000,0.856170,1.576843e-01,1.000000,3.663150e-02
25–34,1.0,1.000000,1.000000,1.000000,1.000000e+00,1.000000,6.073233e-06
35–44,1.0,0.856170,1.000000,1.000000,1.000000e+00,1.000000,2.369163e-06
45–54,1.0,0.157684,1.000000,1.000000,1.000000e+00,0.666833,5.394450e-08
55–64,1.0,1.000000,1.000000,1.000000,6.668331e-01,1.000000,4.389767e-03
65+,1.0,0.036631,0.000006,0.000002,5.394450e-08,0.004390,1.000000e+00


### Interpretation

The post-hoc Dunn's test with Bonferroni correction was used to identify which age-group pairs contributed to the statistically significant Kruskal–Wallis result.

The 65+ age group showed statistically significant differences in transaction activity compared with the 18–24, 25–34, 35–44, 45–54, and 55–64 age groups.

The remaining pairwise comparisons did not produce statistically significant differences after the Bonferroni correction.

Descriptively, the 65+ group also had the lowest median transaction count at 182 transactions per account, compared with 205–221 transactions across the other age groups.

These results indicate that the observed difference in transaction activity is concentrated primarily in comparisons involving the 65+ group. However, statistical significance does not establish practical or commercial importance, and the analysis does not establish that age causes differences in transaction activity.

In [172]:
age_group_epsilon_squared = (
    age_group_kruskal_result.statistic - 7 + 1
) / (
    len(age_group_transaction_activity) - 7
)

print(
    "Kruskal-Wallis epsilon-squared:",
    round(age_group_epsilon_squared, 4)
)

Kruskal-Wallis epsilon-squared: 0.0087


### Effect Size

The Kruskal–Wallis epsilon-squared effect size is 0.0087.

This indicates a small overall effect of customer age group on transaction activity. Although the Kruskal–Wallis test found statistically significant differences between age groups, the effect size suggests that age group explains only a relatively small proportion of the variation in transaction activity.

This distinction is important because the large number of observations can produce statistically significant results even when the practical magnitude of the differences is limited.

Overall, the analysis provides evidence of statistically detectable differences in transaction activity, particularly involving the 65+ age group, but the overall effect is small. The results should therefore be treated as a descriptive finding rather than evidence that age is a major driver of transaction behaviour.

### 4.12 Statistical Analysis Summary

This section summarizes the main statistical findings from the exploratory analysis.

The results combine descriptive statistics, correlation analysis, hypothesis testing, post-hoc comparisons, and effect-size measures.

The objective is to distinguish statistically detectable relationships from findings that may have greater practical or business relevance.

In [173]:
statistical_summary = pd.DataFrame({
    "analysis": [
        "Transaction Amount Distribution",
        "Transaction Amount Percentiles",
        "Transaction Count vs Total Transaction Value",
        "Transaction Amount by Transaction Type",
        "Kruskal-Wallis: Transaction Type",
        "Dunn's Test: Transaction Type",
        "Transaction Count vs Average Transaction Amount",
        "Transaction Count vs Negative-Balance Rate",
        "Effect Size: Transaction Type",
        "Age Group Transaction Activity",
        "Dunn's Test: Age Groups",
        "Effect Size: Age Group"
    ],
    "key_result": [
        "Mean = 5,924.15; Median = 2,100.00",
        "Median = 2,100; 75th percentile = 6,800; 99th percentile = 45,454.81",
        "Pearson r = 0.611",
        "Cash Withdrawal has the highest median; Credit shows the strongest positive skew",
        "H = 31,591.155; p < reporting precision",
        "All transaction-type pairs statistically differ after Bonferroni correction",
        "Pearson r = 0.076",
        "Pearson r = -0.201",
        "ε² = 0.0299",
        "45–54 has the highest mean; 65+ has the lowest mean",
        "65+ differs significantly from 18–24, 25–34, 35–44, 45–54, and 55–64",
        "ε² = 0.0087"
    ],
    "interpretation": [
        "Transaction amounts are positively skewed.",
        "The upper tail contains substantially larger transactions.",
        "More transactions are associated with higher cumulative transaction value.",
        "Transaction-size distributions differ across transaction types.",
        "Transaction amount distributions differ statistically across transaction types.",
        "All three transaction-type pairs show statistically detectable differences.",
        "Transaction frequency has little linear association with average transaction size.",
        "The relationship is weak and negative.",
        "Transaction type explains a relatively small proportion of ranked variation.",
        "Transaction activity differs statistically across some age groups.",
        "The main statistically significant differences involve the 65+ group.",
        "Age group has a small overall effect on transaction activity."
    ]
})

statistical_summary

,analysis,key_result,interpretation
0,Transaction Amount Distribution,"Mean = 5,924.15; Median = 2,100.00",Transaction amounts are positively skewed.
1,Transaction Amount Percentiles,"Median = 2,100; 75th percentile = 6,800; 99th ...",The upper tail contains substantially larger t...
2,Transaction Count vs Total Transaction Value,Pearson r = 0.611,More transactions are associated with higher c...
3,Transaction Amount by Transaction Type,Cash Withdrawal has the highest median; Credit...,Transaction-size distributions differ across t...
4,Kruskal-Wallis: Transaction Type,"H = 31,591.155; p < reporting precision",Transaction amount distributions differ statis...
5,Dunn's Test: Transaction Type,All transaction-type pairs statistically diffe...,All three transaction-type pairs show statisti...
6,Transaction Count vs Average Transaction Amount,Pearson r = 0.076,Transaction frequency has little linear associ...
7,Transaction Count vs Negative-Balance Rate,Pearson r = -0.201,The relationship is weak and negative.
8,Effect Size: Transaction Type,ε² = 0.0299,Transaction type explains a relatively small p...
9,Age Group Transaction Activity,45–54 has the highest mean; 65+ has the lowest...,Transaction activity differs statistically acr...


In [174]:
statistical_summary_export = statistical_summary[
    [
        "analysis",
        "key_result",
        "interpretation"
    ]
].copy()

statistical_summary_export

,analysis,key_result,interpretation
0,Transaction Amount Distribution,"Mean = 5,924.15; Median = 2,100.00",Transaction amounts are positively skewed.
1,Transaction Amount Percentiles,"Median = 2,100; 75th percentile = 6,800; 99th ...",The upper tail contains substantially larger t...
2,Transaction Count vs Total Transaction Value,Pearson r = 0.611,More transactions are associated with higher c...
3,Transaction Amount by Transaction Type,Cash Withdrawal has the highest median; Credit...,Transaction-size distributions differ across t...
4,Kruskal-Wallis: Transaction Type,"H = 31,591.155; p < reporting precision",Transaction amount distributions differ statis...
5,Dunn's Test: Transaction Type,All transaction-type pairs statistically diffe...,All three transaction-type pairs show statisti...
6,Transaction Count vs Average Transaction Amount,Pearson r = 0.076,Transaction frequency has little linear associ...
7,Transaction Count vs Negative-Balance Rate,Pearson r = -0.201,The relationship is weak and negative.
8,Effect Size: Transaction Type,ε² = 0.0299,Transaction type explains a relatively small p...
9,Age Group Transaction Activity,45–54 has the highest mean; 65+ has the lowest...,Transaction activity differs statistically acr...
